# A Machine Learning Approach to Predict Fraud
### XYZ Cybersecurity — Fraud & Cyber-Attack Detection Engine

**Course assignment (25 marks) — end-to-end supervised learning case study**

---

**Business context.** XYZ Cybersecurity is an early-stage startup that detects and blocks online
threats for digital-banking and e-commerce clients. This notebook builds, evaluates and audits a
machine-learning model that flags **fraudulent transactions / account-takeover attacks** in near
real time, so that a small team of human analysts can review the riskiest events first.

**Why this is hard.**

| Challenge | Consequence for modelling |
|---|---|
| Extreme class imbalance (~1-2% fraud) | Accuracy is useless; we optimise **PR-AUC / recall at usable precision** |
| Asymmetric costs (a missed fraud costs the transaction value, a false alarm costs analyst time) | We select the decision **threshold by expected cost**, not by 0.5 |
| Fraud is **adversarial and non-stationary** | Time-ordered validation, drift monitoring, retraining plan |
| Signal lives in *behavioural context*, not in a single row | Feature engineering: velocity, device novelty, deviation from the customer's own baseline |
| Decisions affect real customers | Fairness, privacy and explainability audit (Section 6) |

---

### Marks map — where each requirement is answered

| # | Requirement | Marks | Notebook section |
|---|---|---|---|
| 1 | Data understanding & preparation incl. feature engineering | 3 | [Section 1](#sec1) |
| 2 | Model selection | 3 | [Section 2](#sec2) |
| 3 | Performance measurement | 3 | [Section 3](#sec3) |
| 4 | Hyperparameter tuning | 5 | [Section 4](#sec4) |
| 5 | Extra features and considerations | 3 | [Section 5](#sec5) |
| 6 | AI ethics consideration | 5 | [Section 6](#sec6) |
| 7 | Documentation & code quality | 3 | [Section 7](#sec7) |

---

### How to run

1. Open in Google Colab → `Runtime` → `Run all` (nothing else to configure).
2. The loader in Section 1.1 **first looks for `Dataset1.csv`, `Dataset2.csv`, `Dataset3.csv` in
   `MyDrive`**. If they are present they are used; if they are not, a **fully reproducible
   simulator** generates an equivalent three-table fraud dataset so the notebook always runs
   end-to-end. The provenance is printed so the reader always knows which path was taken.
3. Every narrative conclusion that quotes a number is **generated at run time from the computed
   results**, so the text can never disagree with the tables.
4. Expected runtime on a standard Colab CPU runtime: **~10-25 minutes** (the randomised
   hyper-parameter search in Section 4 dominates). Optional extras (`xgboost`, `imbalanced-learn`,
   `shap`) are used automatically if present and skipped cleanly if not.

*Reproducibility: single global seed (`RANDOM_STATE = 42`), fixed splits, versions printed below.*

## 0. Environment, imports and reproducibility

In [ ]:
# ============================================================================
# 0.1  Imports. Core stack only; optional extras are imported defensively so
#      that the notebook never dies on a missing package.
# ============================================================================
import json
import os
import platform
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*does not have valid feature names.*")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: "{:,.4f}".format(v))
plt.rcParams.update({
    "figure.figsize": (9.5, 4.2),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.30,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

OPTIONAL: Dict[str, str] = {}

try:                                                    # gradient boosting alternative
    import xgboost as xgb
    OPTIONAL["xgboost"] = xgb.__version__
except Exception:
    xgb = None

try:                                                    # resampling for imbalance
    import imblearn
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    OPTIONAL["imbalanced-learn"] = imblearn.__version__
except Exception:
    SMOTE, ImbPipeline = None, None

try:                                                    # model explainability
    import shap
    OPTIONAL["shap"] = shap.__version__
except Exception:
    shap = None

try:                                                    # model persistence
    import joblib
    OPTIONAL["joblib"] = joblib.__version__
except Exception:
    joblib = None

print("=" * 74)
print("ENVIRONMENT")
print("=" * 74)
print("{:<18}{}".format("python", sys.version.split()[0]))
print("{:<18}{}".format("platform", platform.platform()))
for name, mod in [("numpy", np), ("pandas", pd), ("scikit-learn", sklearn),
                  ("matplotlib", matplotlib)]:
    print("{:<18}{}".format(name, mod.__version__))
for name, ver in OPTIONAL.items():
    print("{:<18}{}".format(name, ver))
missing = [n for n in ["xgboost", "imbalanced-learn", "shap", "joblib"] if n not in OPTIONAL]
print("{:<18}{}".format("optional missing", missing if missing else "none"))
print("=" * 74)

In [ ]:
# ============================================================================
# 0.2  Single source of truth for every tunable constant + global seeding.
#      Keeping configuration in one immutable object (instead of magic numbers
#      sprinkled through the code) is what makes the run reproducible and the
#      notebook easy to re-point at a different dataset.
# ============================================================================
RANDOM_STATE = 42


@dataclass(frozen=True)
class Config:
    """Experiment configuration for the fraud-detection pipeline."""

    # --- reproducibility -----------------------------------------------------
    random_state: int = RANDOM_STATE

    # --- data location (real files take priority over the simulator) ---------
    drive_dir: str = "/content/drive/MyDrive"
    transaction_file: str = "Dataset1.csv"
    customer_file: str = "Dataset2.csv"
    device_file: str = "Dataset3.csv"

    # --- schema --------------------------------------------------------------
    target: str = "is_fraud"
    time_col: str = "timestamp"

    # --- simulator size (only used when the CSVs are absent) -----------------
    n_transactions: int = 60_000
    n_customers: int = 3_000
    n_devices: int = 4_500
    n_compromise_events: int = 170          # account-takeover bursts
    n_single_shot_fraud: int = 330          # lone-wolf frauds
    label_noise: float = 0.03               # frauds never reported -> label 0
    sim_start: str = "2024-01-01"
    sim_days: int = 365

    # --- chronological split -------------------------------------------------
    train_frac: float = 0.60
    val_frac: float = 0.20                  # test = remaining 20%

    # --- business economics used for threshold selection --------------------
    review_cost: float = 8.0                # analyst cost per alert (USD)
    recovery_rate: float = 0.25             # share of fraud value recovered
    friction_cost: float = 3.0              # goodwill cost of a wrong block
    analyst_capacity: float = 0.01          # can review 1% of traffic per day

    # --- cross-validation ----------------------------------------------------
    cv_splits: int = 4
    search_iterations: int = 30

    # --- output --------------------------------------------------------------
    artifact_dir: str = "artifacts"


CFG = Config()
os.makedirs(CFG.artifact_dir, exist_ok=True)
np.random.seed(CFG.random_state)

print("CONFIGURATION")
print(json.dumps(asdict(CFG), indent=2))

In [ ]:
# ============================================================================
# 0.3  Small, reusable helpers used throughout the notebook.
# ============================================================================
def section(title: str, char: str = "=") -> None:
    """Print a consistent section banner (keeps long outputs readable)."""
    print(char * 78)
    print(title.upper())
    print(char * 78)


def show(df: pd.DataFrame, n: int = 5, title: Optional[str] = None) -> None:
    """Display a dataframe head with an optional caption (Colab-friendly)."""
    if title:
        print("\n>>> {}   shape={}".format(title, df.shape))
    try:
        from IPython.display import display
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string())


def pct(x: float) -> str:
    """Format a proportion as a percentage string."""
    return "{:.3f}%".format(100.0 * x)


def money(x: float) -> str:
    """Format a USD amount."""
    return "${:,.0f}".format(x)


def make_ohe():
    """OneHotEncoder that returns a dense array across scikit-learn versions."""
    from sklearn.preprocessing import OneHotEncoder
    try:                                     # scikit-learn >= 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                             min_frequency=10)
    except TypeError:                        # older releases
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def safe_estimator(cls, **kwargs):
    """Instantiate `cls`, dropping keyword arguments the installed version
    does not support (e.g. `class_weight` on older HistGradientBoosting)."""
    try:
        return cls(**kwargs)
    except TypeError as exc:
        bad = [k for k in list(kwargs) if k in str(exc)]
        for k in bad:
            kwargs.pop(k, None)
        print("  note: {} does not support {} -> dropped".format(cls.__name__, bad))
        return cls(**kwargs)


print("Helpers ready:", [f.__name__ for f in (section, show, pct, money, make_ohe,
                                              safe_estimator)])

<a name="sec1"></a>
# 1. Data Understanding and Preparation  *(3 marks)*

The engine consumes **three related tables**, which is how fraud data really arrives — the event
stream is useless without the entity context around it:

| Table | Grain | Role |
|---|---|---|
| `Dataset1.csv` — **transactions** | one row per payment | the event stream + the `is_fraud` label |
| `Dataset2.csv` — **customers** | one row per customer | KYC / account context (tenure, limit, segment) |
| `Dataset3.csv` — **devices** | one row per device | technical fingerprint (OS, VPN/proxy, rooted) |

**1.1** load (Drive first, reproducible simulator as fallback) → **1.2** merge & referential audit →
**1.3** profiling / EDA → **1.4** data-quality cleaning → **1.5** feature engineering →
**1.6** leakage-safe chronological split.

### 1.1 Loading the data

The cell below mounts Drive and uses the three CSVs **if they exist**. Otherwise it calls a seeded
simulator that produces the same schema, so this notebook is self-contained and re-runnable by a
marker who does not have the original files. The generative process is documented in code because
knowing *how fraud enters the data* is exactly what motivates the features in Section 1.5.

In [ ]:
# ============================================================================
# 1.1a  Try to mount Google Drive (no-op outside Colab).
# ============================================================================
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:                    # already mounted / not authorised
        print("Drive mount skipped:", exc)
else:
    print("Not running in Colab -> skipping Drive mount.")

CANDIDATE_PATHS = {
    "transactions": os.path.join(CFG.drive_dir, CFG.transaction_file),
    "customers": os.path.join(CFG.drive_dir, CFG.customer_file),
    "devices": os.path.join(CFG.drive_dir, CFG.device_file),
}
FOUND = {k: p for k, p in CANDIDATE_PATHS.items() if os.path.exists(p)}

section("data source discovery")
for name, path in CANDIDATE_PATHS.items():
    print("  {:<14} {:<45} {}".format(name, path,
                                      "FOUND" if name in FOUND else "not found"))
USE_REAL_FILES = len(FOUND) == len(CANDIDATE_PATHS)
print("\n-> using {}".format("the CSV files from Drive" if USE_REAL_FILES
                             else "the built-in reproducible simulator"))

In [ ]:
# ============================================================================
# 1.1b  Reproducible fraud-data simulator.
#
#  Domain assumptions encoded here (they mirror published card-fraud patterns
#  and are what the model must learn to recover):
#    * ~1.5% of transactions are fraudulent, so the classes are very imbalanced
#    * most fraud arrives as an ACCOUNT-TAKEOVER BURST: several fast, larger,
#      night-time payments from a device the customer has never used, often
#      behind a VPN/proxy and often cross-border
#    * a minority is LONE-WOLF fraud that looks almost normal (class overlap,
#      so the problem is not linearly separable and metrics stay realistic)
#    * a share of true fraud is never reported -> LABEL NOISE
#    * the raw feed contains duplicates, missing values, messy strings and
#      impossible amounts -> gives Section 1.4 real work to do
#    * younger customers legitimately churn devices more often, which later
#      produces a measurable FAIRNESS disparity for Section 6
# ============================================================================
CATEGORIES: Dict[str, Tuple[float, float, float, float]] = {
    # name          weight   log-mu  log-sd  fraud attractiveness
    "grocery":      (0.20,   3.30,   0.65,   0.2),
    "restaurant":   (0.14,   3.10,   0.70,   0.3),
    "transport":    (0.10,   2.85,   0.75,   0.4),
    "retail":       (0.13,   3.85,   0.85,   0.8),
    "utilities":    (0.09,   4.20,   0.55,   0.2),
    "healthcare":   (0.05,   4.30,   0.80,   0.3),
    "electronics":  (0.06,   5.40,   0.75,   1.6),
    "travel":       (0.05,   5.75,   0.80,   1.4),
    "gaming":       (0.07,   3.05,   0.90,   1.2),
    "digital_gift": (0.04,   4.10,   0.80,   2.0),
    "crypto":       (0.02,   6.00,   0.85,   2.4),
    "gambling":     (0.05,   4.90,   0.95,   1.8),
}
HOME_COUNTRIES = ["MY", "SG", "ID", "TH", "VN"]
LOW_RISK_FOREIGN = np.array(["SG", "TH", "ID", "VN", "MY", "JP", "AU"])
HIGH_RISK_FOREIGN = np.array(["RU", "CN", "NG", "UA", "US", "BR"])

_LEGIT_HOUR_W = np.array([0.6, 0.4, 0.3, 0.3, 0.4, 0.8, 1.6, 3.0, 4.6, 5.4, 5.8, 6.2,
                          6.6, 6.0, 5.6, 5.4, 5.6, 6.0, 6.4, 6.0, 4.8, 3.4, 2.0, 1.2])
_FRAUD_HOUR_W = np.array([4.6, 5.0, 5.2, 4.8, 4.0, 3.0, 2.0, 1.4, 1.0, 0.9, 0.9, 1.0,
                          1.1, 1.0, 1.0, 1.1, 1.3, 1.6, 2.0, 2.6, 3.2, 3.8, 4.2, 4.4])
_LEGIT_HOUR_W = _LEGIT_HOUR_W / _LEGIT_HOUR_W.sum()
_FRAUD_HOUR_W = _FRAUD_HOUR_W / _FRAUD_HOUR_W.sum()


def simulate_customers(n: int, rng: np.random.Generator) -> pd.DataFrame:
    """Create the customer/KYC table (`Dataset2.csv` equivalent)."""
    age = np.clip(rng.normal(41, 14, n), 18, 88).round().astype(int)
    tenure_days = rng.integers(10, 3_500, n)
    credit_limit = np.round(np.clip(rng.lognormal(8.5, 0.55, n), 500, 60_000), -2)
    df = pd.DataFrame({
        "customer_id": ["C{:06d}".format(i) for i in range(n)],
        "age": age,
        "gender": rng.choice(["F", "M", "U"], n, p=[0.47, 0.49, 0.04]),
        "home_country": rng.choice(HOME_COUNTRIES, n, p=[0.55, 0.15, 0.14, 0.09, 0.07]),
        "region": rng.choice(["Central", "North", "South", "East", "Borneo"], n,
                             p=[0.38, 0.20, 0.18, 0.13, 0.11]),
        "account_type": rng.choice(["standard", "premium", "business"], n,
                                   p=[0.72, 0.20, 0.08]),
        "signup_date": pd.Timestamp(CFG.sim_start) - pd.to_timedelta(tenure_days, unit="D"),
        "credit_limit": credit_limit,
        "kyc_level": rng.choice(["basic", "full"], n, p=[0.23, 0.77]),
    })
    # realistic gaps in the KYC feed
    gaps = rng.choice(n, size=int(0.02 * n), replace=False)
    df.loc[gaps, "credit_limit"] = np.nan
    return df


def simulate_devices(n: int, rng: np.random.Generator) -> pd.DataFrame:
    """Create the device-fingerprint table (`Dataset3.csv` equivalent)."""
    return pd.DataFrame({
        "device_id": ["D{:06d}".format(i) for i in range(n)],
        "device_type": rng.choice(["android", "ios", "windows", "macos", "other"], n,
                                  p=[0.42, 0.30, 0.17, 0.08, 0.03]),
        "os_major": rng.integers(9, 19, n),
        "device_age_days": rng.integers(0, 1_800, n),
        "is_proxy_vpn": rng.random(n) < 0.07,
        "is_rooted": rng.random(n) < 0.04,
        "screen_class": rng.choice(["small", "medium", "large"], n, p=[0.3, 0.5, 0.2]),
    })


def _draw_block(cust_idx: np.ndarray, timestamps: pd.DatetimeIndex,
                customers: pd.DataFrame, devices: pd.DataFrame,
                primary_device: np.ndarray, rng: np.random.Generator,
                fraudulent: bool) -> pd.DataFrame:
    """Draw a block of transactions for the given customers/timestamps.

    The only difference between the fraudulent and the legitimate generator is
    the *tilt* of the same distributions (bigger amounts, unknown devices,
    cross-border, night-time, risky merchant categories) - never a hard rule,
    which keeps the two classes genuinely overlapping.
    """
    m = len(cust_idx)
    cust = customers.iloc[cust_idx].reset_index(drop=True)
    names = list(CATEGORIES)

    weights = np.array([CATEGORIES[c][0] * (CATEGORIES[c][3] if fraudulent else 1.0)
                        for c in names])
    weights = weights / weights.sum()
    category = rng.choice(names, size=m, p=weights)
    mu = np.array([CATEGORIES[c][1] for c in category])
    sd = np.array([CATEGORIES[c][2] for c in category])

    limit = cust["credit_limit"].fillna(cust["credit_limit"].median()).to_numpy()
    wealth = np.log1p(limit) / np.log(10_000.0)
    amount = np.exp(rng.normal(mu, sd) + 0.25 * (wealth - 1.0))
    if fraudulent:
        amount = amount * rng.lognormal(0.45, 0.45, m)
    amount = np.round(np.clip(amount, 1.0, 25_000.0), 2)

    # ---- device: fraud nearly always comes from an unfamiliar device --------
    age = cust["age"].to_numpy()
    p_new = np.where(np.full(m, fraudulent), 0.85, 0.05 + 0.09 * (age < 30))
    use_new = rng.random(m) < p_new
    dev_idx = primary_device[cust_idx].copy()
    n_new = int(use_new.sum())
    if n_new:
        if fraudulent:
            risky = np.flatnonzero(devices["is_proxy_vpn"].to_numpy()
                                   | devices["is_rooted"].to_numpy())
            pool = risky if len(risky) else np.arange(len(devices))
        else:
            pool = np.arange(len(devices))
        dev_idx[use_new] = rng.choice(pool, size=n_new)

    # ---- geography ---------------------------------------------------------
    home = cust["home_country"].to_numpy().astype(object)
    ip_country, merch_country = home.copy(), home.copy()
    for target, p_foreign in ((ip_country, 0.55 if fraudulent else 0.05),
                              (merch_country, 0.45 if fraudulent else 0.07)):
        far = rng.random(m) < p_foreign
        k = int(far.sum())
        if k:
            target[far] = rng.choice(HIGH_RISK_FOREIGN if fraudulent
                                     else LOW_RISK_FOREIGN, size=k)

    # ---- channel: age-correlated for legitimate traffic --------------------
    if fraudulent:
        channel = rng.choice(["web", "app", "pos"], m, p=[0.62, 0.31, 0.07])
    else:
        young = age < 35
        channel = np.where(
            young,
            rng.choice(["web", "app", "pos"], m, p=[0.24, 0.58, 0.18]),
            rng.choice(["web", "app", "pos"], m, p=[0.34, 0.28, 0.38]),
        )

    return pd.DataFrame({
        "customer_id": cust["customer_id"].to_numpy(),
        "device_id": devices["device_id"].to_numpy()[dev_idx],
        "timestamp": timestamps,
        "amount": amount,
        "merchant_category": category,
        "channel": channel,
        "ip_country": ip_country,
        "merchant_country": merch_country,
        "card_present": np.where(channel == "pos", rng.random(m) < 0.9,
                                 rng.random(m) < 0.02),
        "auth_3ds": rng.random(m) < (0.12 if fraudulent else 0.42),
        "is_fraud": int(fraudulent),
    })


def _random_timestamps(n: int, rng: np.random.Generator,
                       fraud_hours: bool = False) -> pd.DatetimeIndex:
    """Uniform days over the simulation window with a realistic hour profile."""
    day = rng.integers(0, CFG.sim_days, n)
    hour = rng.choice(24, size=n, p=_FRAUD_HOUR_W if fraud_hours else _LEGIT_HOUR_W)
    minute, second = rng.integers(0, 60, n), rng.integers(0, 60, n)
    return (pd.Timestamp(CFG.sim_start)
            + pd.to_timedelta(day, unit="D")
            + pd.to_timedelta(hour, unit="h")
            + pd.to_timedelta(minute, unit="m")
            + pd.to_timedelta(second, unit="s"))


def simulate_fraud_dataset(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build the three-table dataset (transactions, customers, devices)."""
    rng = np.random.default_rng(cfg.random_state)
    customers = simulate_customers(cfg.n_customers, rng)
    devices = simulate_devices(cfg.n_devices, rng)
    primary_device = rng.choice(len(devices), size=len(customers))

    # activity is power-law distributed: a few customers transact constantly
    activity = rng.pareto(1.6, len(customers)) + 1.0
    activity = activity / activity.sum()

    # ---- 1. legitimate traffic --------------------------------------------
    n_fraud_burst_est = cfg.n_compromise_events * 5
    n_legit = cfg.n_transactions - n_fraud_burst_est - cfg.n_single_shot_fraud
    legit_idx = rng.choice(len(customers), size=n_legit, p=activity)
    legit = _draw_block(legit_idx, _random_timestamps(n_legit, rng),
                        customers, devices, primary_device, rng, fraudulent=False)

    # ---- 2. account-takeover bursts ---------------------------------------
    victims = rng.choice(len(customers), size=cfg.n_compromise_events, p=activity)
    burst_cust, burst_ts = [], []
    for victim in victims:
        k = int(rng.integers(2, 9))
        start = _random_timestamps(1, rng, fraud_hours=True)[0]
        offsets = np.cumsum(rng.exponential(8.0, k))          # minutes apart
        burst_cust.extend([victim] * k)
        burst_ts.extend([start + pd.to_timedelta(o, unit="m") for o in offsets])
    bursts = _draw_block(np.array(burst_cust), pd.DatetimeIndex(burst_ts),
                         customers, devices, primary_device, rng, fraudulent=True)

    # ---- 3. lone-wolf fraud (hard cases) ----------------------------------
    lone_idx = rng.choice(len(customers), size=cfg.n_single_shot_fraud, p=activity)
    lone = _draw_block(lone_idx, _random_timestamps(cfg.n_single_shot_fraud, rng,
                                                    fraud_hours=True),
                       customers, devices, primary_device, rng, fraudulent=True)

    txn = pd.concat([legit, bursts, lone], ignore_index=True)
    txn = txn.sort_values("timestamp").reset_index(drop=True)

    # ---- 4. label noise: some real fraud is never reported -----------------
    fraud_rows = np.flatnonzero(txn["is_fraud"].to_numpy() == 1)
    unreported = rng.choice(fraud_rows,
                            size=int(cfg.label_noise * len(fraud_rows)),
                            replace=False)
    txn.loc[unreported, "is_fraud"] = 0

    # ---- 5. realistic data-quality defects --------------------------------
    txn["txn_id"] = ["T{:08d}".format(i) for i in
                     rng.permutation(len(txn))]
    dupes = txn.sample(n=300, random_state=cfg.random_state)          # replayed rows
    txn = pd.concat([txn, dupes], ignore_index=True)

    n = len(txn)
    txn.loc[rng.choice(n, int(0.010 * n), replace=False), "merchant_category"] = np.nan
    txn.loc[rng.choice(n, int(0.005 * n), replace=False), "channel"] = np.nan
    txn.loc[rng.choice(n, int(0.004 * n), replace=False), "amount"] = np.nan
    txn.loc[rng.choice(n, 25, replace=False), "amount"] = -1.0        # impossible
    messy = rng.choice(n, int(0.03 * n), replace=False)               # messy strings
    txn.loc[messy, "channel"] = txn.loc[messy, "channel"].astype(str).str.upper()

    cols = ["txn_id", "customer_id", "device_id", "timestamp", "amount",
            "merchant_category", "channel", "ip_country", "merchant_country",
            "card_present", "auth_3ds", "is_fraud"]
    return txn[cols], customers, devices


print("Simulator defined: simulate_fraud_dataset() ->",
      "(transactions, customers, devices)")

In [ ]:
# ============================================================================
# 1.1c  Load: real CSVs when available, otherwise the simulator.
# ============================================================================
def _standardise_columns(df: pd.DataFrame) -> pd.DataFrame:
    """snake_case the column names of an externally supplied CSV."""
    df = df.copy()
    df.columns = (df.columns.str.strip().str.replace(r"\s+", "_", regex=True)
                  .str.replace(r"(?<=[a-z0-9])(?=[A-Z])", "_", regex=True).str.lower())
    return df


def _detect(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    """Return the first column whose name matches one of `candidates`."""
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


if USE_REAL_FILES:
    DATA_SOURCE = "Google Drive CSVs"
    txn_raw = _standardise_columns(pd.read_csv(FOUND["transactions"]))
    cust_raw = _standardise_columns(pd.read_csv(FOUND["customers"]))
    dev_raw = _standardise_columns(pd.read_csv(FOUND["devices"]))

    # align the label / time column names with the rest of the notebook
    label = _detect(txn_raw, ["is_fraud", "isfraud", "fraud", "class", "target", "label"])
    if label and label != CFG.target:
        txn_raw = txn_raw.rename(columns={label: CFG.target})
    tcol = _detect(txn_raw, ["timestamp", "trans_date_trans_time", "datetime", "date",
                             "time", "step"])
    if tcol and tcol != CFG.time_col:
        txn_raw = txn_raw.rename(columns={tcol: CFG.time_col})
else:
    DATA_SOURCE = "built-in simulator (seed={})".format(CFG.random_state)
    txn_raw, cust_raw, dev_raw = simulate_fraud_dataset(CFG)

section("raw tables loaded from {}".format(DATA_SOURCE))
for name, frame in [("transactions", txn_raw), ("customers", cust_raw),
                    ("devices", dev_raw)]:
    print("  {:<14} rows={:>7,}  cols={:>3}   {}".format(
        name, len(frame), frame.shape[1], list(frame.columns)[:6]))

show(txn_raw, 5, "transactions (Dataset1)")
show(cust_raw, 5, "customers (Dataset2)")
show(dev_raw, 5, "devices (Dataset3)")

### 1.2 Joining the three tables + referential-integrity audit

A silent join failure is one of the most common causes of a "mysteriously bad" fraud model, so the
merge is explicitly audited: row count must be preserved (left join on a unique key) and orphan
foreign keys are counted rather than assumed to be zero.

In [ ]:
# ============================================================================
# 1.2  Merge transactions <- customers <- devices, then audit the join.
# ============================================================================
def assemble(txn: pd.DataFrame, cust: pd.DataFrame,
             dev: pd.DataFrame) -> pd.DataFrame:
    """Left-join the dimension tables onto the transaction stream.

    Left joins are used deliberately: an unknown customer or an unseen device
    is itself a fraud signal, so those rows must be kept (with NaNs)
    rather than dropped by an inner join.
    """
    out = txn.copy()
    for dim, key in ((cust, "customer_id"), (dev, "device_id")):
        if key in out.columns and key in dim.columns:
            before = len(out)
            dim_unique = dim.drop_duplicates(subset=[key])
            out = out.merge(dim_unique, on=key, how="left")
            assert len(out) == before, "join fan-out on {}".format(key)
    return out


df = assemble(txn_raw, cust_raw, dev_raw)

section("join audit")
print("transactions in : {:,}".format(len(txn_raw)))
print("rows after join : {:,}  (must be identical)".format(len(df)))
for key, dim in (("customer_id", cust_raw), ("device_id", dev_raw)):
    if key in df.columns and key in dim.columns:
        orphans = (~df[key].isin(dim[key])).sum()
        print("  orphan {:<12} {:>6,}  ({} of transactions)".format(
            key + ":", orphans, pct(orphans / len(df))))
print("\nmerged shape    :", df.shape)
show(df, 4, "merged analytical base table")

### 1.3 Data understanding — profiling and exploratory analysis

Four questions drive everything downstream: *how imbalanced is the target*, *where are the missing
values*, *which raw signals separate the classes*, and *is the target rate stable over time*
(if it is not, we must validate chronologically).

In [ ]:
# ============================================================================
# 1.3a  Structure, dtypes, missingness, duplicates, target balance.
# ============================================================================
section("1. schema and dtypes")
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
    "pct_missing": (100 * df.isna().mean()).round(3),
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None
                for c in df.columns],
})
show(schema.sort_values("pct_missing", ascending=False), len(schema), "column profile")

section("2. duplicates")
dup_rows = df.duplicated().sum()
dup_ids = df["txn_id"].duplicated().sum() if "txn_id" in df.columns else 0
print("fully duplicated rows      : {:,}".format(dup_rows))
print("duplicated transaction ids : {:,}".format(dup_ids))

section("3. target balance")
counts = df[CFG.target].value_counts().sort_index()
rate = df[CFG.target].mean()
print("legitimate (0): {:>8,}".format(int(counts.get(0, 0))))
print("fraud      (1): {:>8,}".format(int(counts.get(1, 0))))
print("fraud rate    : {}   -> imbalance ratio 1:{:.0f}".format(
    pct(rate), (1 - rate) / max(rate, 1e-12)))
print("\nIMPLICATION: a model that predicts 'never fraud' already scores"
      " {} accuracy.\n             Accuracy is therefore not a usable metric"
      " (see Section 3).".format(pct(1 - rate)))

section("4. numeric summary")
show(df.select_dtypes(include=[np.number]).describe().T, 20, "describe()")

In [ ]:
# ============================================================================
# 1.3b  Visual EDA: imbalance, amount separation, temporal and categorical risk.
# ============================================================================
tmp = df.copy()
tmp[CFG.time_col] = pd.to_datetime(tmp[CFG.time_col], errors="coerce")
tmp["_hour"] = tmp[CFG.time_col].dt.hour
tmp["_month"] = tmp[CFG.time_col].dt.to_period("M").astype(str)

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.2))

# (a) class imbalance -------------------------------------------------------
ax = axes[0, 0]
bars = ax.bar(["legitimate", "fraud"],
              [int((tmp[CFG.target] == 0).sum()), int((tmp[CFG.target] == 1).sum())],
              color=["#4C78A8", "#E45756"])
ax.set_yscale("log")
ax.set_title("(a) Class imbalance (log scale)")
ax.set_ylabel("transactions")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), "{:,}".format(int(b.get_height())),
            ha="center", va="bottom", fontsize=9)

# (b) amount distribution by class -----------------------------------------
ax = axes[0, 1]
amt = tmp["amount"].where(tmp["amount"] > 0)
bins = np.logspace(0, np.log10(max(amt.max(), 10)), 45)
ax.hist(amt[tmp[CFG.target] == 0].dropna(), bins=bins, density=True, alpha=0.6,
        label="legitimate", color="#4C78A8")
ax.hist(amt[tmp[CFG.target] == 1].dropna(), bins=bins, density=True, alpha=0.6,
        label="fraud", color="#E45756")
ax.set_xscale("log")
ax.set_title("(b) Transaction amount by class (overlapping!)")
ax.set_xlabel("amount (log)")
ax.legend()

# (c) fraud rate by hour of day --------------------------------------------
ax = axes[1, 0]
by_hour = tmp.groupby("_hour")[CFG.target].mean() * 100
ax.plot(by_hour.index, by_hour.to_numpy(), marker="o", color="#E45756")
ax.axhline(tmp[CFG.target].mean() * 100, ls="--", c="grey", label="overall rate")
ax.set_title("(c) Fraud rate by hour of day")
ax.set_xlabel("hour")
ax.set_ylabel("fraud rate (%)")
ax.set_xticks(range(0, 24, 2))
ax.legend()

# (d) fraud rate by merchant category --------------------------------------
ax = axes[1, 1]
by_cat = (tmp.groupby("merchant_category")[CFG.target].agg(["mean", "size"])
          .sort_values("mean", ascending=True))
ax.barh(by_cat.index.astype(str), by_cat["mean"] * 100, color="#72B7B2")
ax.set_title("(d) Fraud rate by merchant category")
ax.set_xlabel("fraud rate (%)")
plt.tight_layout()
plt.show()

section("temporal stability of the target (drives the split strategy)")
monthly = tmp.groupby("_month")[CFG.target].agg(n="size", frauds="sum", rate="mean")
monthly["rate"] = (monthly["rate"] * 100).round(3)
print(monthly.to_string())

In [ ]:
# ============================================================================
# 1.3c  Correlation structure of the numeric columns (redundancy check).
# ============================================================================
num = df.select_dtypes(include=[np.number]).copy()
num[CFG.target] = df[CFG.target]
corr = num.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(7.4, 6.2))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, "{:.2f}".format(corr.iloc[i, j]), ha="center", va="center",
                fontsize=7, color="black")
ax.set_title("Pearson correlation of raw numeric columns")
ax.grid(False)
fig.colorbar(im, shrink=0.8)
plt.tight_layout()
plt.show()

print("Linear correlation of the raw columns with the target (weak by design):")
print(corr[CFG.target].drop(CFG.target).sort_values(key=abs, ascending=False)
      .round(4).to_string())
print("\nWeak univariate correlation is exactly why we need (i) engineered"
      " behavioural\nfeatures and (ii) non-linear models.")

### 1.4 Data preparation — cleaning the defects found above

Every action below is *reported with a count* rather than applied silently, and any step that needs
a statistic (e.g. a median) computes it **only from the training window** so that no information
from the future leaks into the past.

In [ ]:
# ============================================================================
# 1.4  Cleaning. Returns the cleaned frame plus an auditable action log.
# ============================================================================
ID_LIKE = ("txn_id", "customer_id", "device_id")


def clean_transactions(frame: pd.DataFrame,
                       cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Normalise types/strings, remove replayed rows and repair impossible values.

    Parameters
    ----------
    frame : merged analytical base table.
    cfg   : experiment configuration (needs `time_col`, `train_frac`).

    Returns
    -------
    (cleaned frame, action log)
    """
    out, log = frame.copy(), []

    def note(step: str, n: int, detail: str = "") -> None:
        log.append({"step": step, "rows_affected": int(n), "detail": detail})

    # --- 1. parse the event time -------------------------------------------
    out[cfg.time_col] = pd.to_datetime(out[cfg.time_col], errors="coerce")
    bad_time = out[cfg.time_col].isna().sum()
    if bad_time:
        out = out[out[cfg.time_col].notna()]
    note("drop unparseable timestamps", bad_time)

    # --- 2. normalise free-text categoricals -------------------------------
    obj_cols = [c for c in out.select_dtypes(include="object").columns
                if c not in ID_LIKE]
    touched = 0
    for col in obj_cols:
        original = out[col].copy()
        cleaned = out[col].astype(str).str.strip().str.lower()
        cleaned = cleaned.replace({"nan": np.nan, "none": np.nan, "": np.nan,
                                   "null": np.nan})
        touched += int((original.astype(str) != cleaned.astype(str)).sum())
        out[col] = cleaned
    note("normalise case/whitespace in categoricals", touched,
         ", ".join(obj_cols))

    # --- 3. replayed / duplicated events -----------------------------------
    n0 = len(out)
    out = out.drop_duplicates()
    note("drop fully duplicated rows", n0 - len(out))
    if "txn_id" in out.columns:
        n0 = len(out)
        out = out.drop_duplicates(subset=["txn_id"], keep="first")
        note("drop repeated transaction ids", n0 - len(out))

    # --- 4. impossible amounts ---------------------------------------------
    invalid = (out["amount"] <= 0) | (out["amount"].isna())
    note("amount <= 0 or missing -> imputed", invalid.sum())
    out.loc[out["amount"] <= 0, "amount"] = np.nan

    out = out.sort_values(cfg.time_col).reset_index(drop=True)

    # median from the TRAINING WINDOW ONLY (no look-ahead)
    cut = out[cfg.time_col].quantile(cfg.train_frac)
    train_mask = out[cfg.time_col] <= cut
    cust_median = (out.loc[train_mask].groupby("customer_id")["amount"].median())
    global_median = out.loc[train_mask, "amount"].median()
    filled = out["amount"].fillna(out["customer_id"].map(cust_median))
    out["amount"] = filled.fillna(global_median)
    note("amount imputed (customer median -> global median, train window)",
         invalid.sum(), "train-window median = {:.2f}".format(global_median))

    # --- 5. explicit category for missing labels of a category -------------
    for col in ("merchant_category", "channel"):
        if col in out.columns:
            miss = out[col].isna().sum()
            out[col] = out[col].fillna("unknown")
            note("missing {} -> 'unknown'".format(col), miss,
                 "missingness is itself a signal, so it is encoded, not dropped")

    # --- 6. booleans to integers ------------------------------------------
    for col in out.columns:
        if out[col].dtype == bool:
            out[col] = out[col].astype(int)

    # --- 7. extreme amounts: reported, NOT removed -------------------------
    hi = out["amount"].quantile(0.9995)
    note("extreme amounts kept (flagged only)", int((out["amount"] > hi).sum()),
         "high-value outliers are frequently genuine fraud - removing them "
         "would delete the signal; a log transform tames their leverage")

    return out.reset_index(drop=True), pd.DataFrame(log)


df_clean, clean_log = clean_transactions(df, CFG)

section("cleaning action log")
print(clean_log.to_string(index=False))
print("\nrows: {:,} -> {:,}   |   fraud rate: {} -> {}".format(
    len(df), len(df_clean), pct(df[CFG.target].mean()), pct(df_clean[CFG.target].mean())))
print("remaining missing values (handled inside the model pipeline):")
rem = df_clean.isna().sum()
print(rem[rem > 0].to_string() if (rem > 0).any() else "  none")

### 1.5 Feature engineering — the highest-leverage step

A single transaction row is almost uninformative: **USD 900 spent on electronics at 02:00** is
either a crime or a normal Tuesday, and only the *context* decides which. We therefore build four
families of features, all of them computed **strictly from each customer's past** (`shift`/
`rolling` over a time-sorted stream), which is the only way to keep an offline model honest about
what it could actually know at scoring time.

| Family | Features | Fraud hypothesis |
|---|---|---|
| **Temporal** | `hour`, `day_of_week`, `is_night`, `is_weekend`, `days_since_signup` | attacks cluster at night and on fresh accounts |
| **Deviation from own baseline** | `cust_prev_mean_amount`, `amount_vs_cust_mean`, `amount_to_limit`, `log_amount` | fraud spends unlike the real owner |
| **Velocity / burst** | `n_txn_prev_1h/24h/7d`, `amt_sum_prev_24h`, `secs_since_prev_txn`, `is_burst_1h` | card testing and cash-out happen in bursts |
| **Identity & geography** | `is_first_time_device`, `is_new_device`, `device_shared`, `ip_not_home`, `ip_high_risk`, `ip_vs_merchant_mismatch`, `is_proxy_vpn`, `is_rooted`, `auth_3ds` | account takeover means new device, masked IP, cross-border cash-out |

In [ ]:
# ============================================================================
# 1.5a  Behavioural history features (per customer, past-only).
# ============================================================================
def add_behavioural_features(frame: pd.DataFrame, cfg: Config,
                             windows: Sequence[str] = ("1h", "24h", "7d")
                             ) -> pd.DataFrame:
    """Add velocity / baseline-deviation / device-novelty features.

    All values are derived from transactions that occurred **before** the row
    being described (rolling windows exclude the current amount, `shift()` is
    used for the previous device / IP), which guarantees the features are
    reproducible at inference time and free of look-ahead bias.
    """
    parts: List[pd.DataFrame] = []
    for _, grp in frame.groupby("customer_id", sort=False):
        g = grp.sort_values(cfg.time_col).copy()
        amounts = g["amount"].to_numpy(dtype=float)
        series = g.set_index(cfg.time_col)["amount"]

        for win in windows:
            roll = series.rolling(win)
            g["n_txn_prev_" + win] = roll.count().to_numpy() - 1.0
            g["amt_sum_prev_" + win] = roll.sum().to_numpy() - amounts

        n_prev = np.arange(len(g), dtype=float)
        cum_prev = np.cumsum(amounts) - amounts
        prev_mean = np.divide(cum_prev, n_prev, out=np.full(len(g), np.nan),
                              where=n_prev > 0)
        g["cust_txn_seq"] = n_prev
        g["cust_prev_mean_amount"] = prev_mean
        g["amount_vs_cust_mean"] = np.divide(
            amounts, prev_mean, out=np.full(len(g), np.nan),
            where=np.nan_to_num(prev_mean) > 0)
        g["secs_since_prev_txn"] = g[cfg.time_col].diff().dt.total_seconds()

        prev_device = g["device_id"].shift()
        g["is_new_device"] = ((prev_device.notna())
                              & (prev_device != g["device_id"])).astype(int)
        seen: set = set()
        first_time = []
        for dev in g["device_id"]:
            first_time.append(0 if dev in seen else 1)
            seen.add(dev)
        g["is_first_time_device"] = first_time

        prev_ip = g["ip_country"].shift()
        g["ip_country_changed"] = ((prev_ip.notna())
                                   & (prev_ip != g["ip_country"])).astype(int)
        parts.append(g)

    out = pd.concat(parts).sort_values(cfg.time_col).reset_index(drop=True)
    out["is_burst_1h"] = (out["n_txn_prev_1h"] >= 3).astype(int)
    return out


def add_device_sharing_features(frame: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Flag devices that have already been used by *other* customers.

    Fraud rings re-use infrastructure, so a device seen with several distinct
    accounts is a classic ring indicator. Computed in one chronological pass,
    therefore also past-only.
    """
    out = frame.sort_values(cfg.time_col).reset_index(drop=True).copy()
    seen_customers: Dict[Any, set] = {}
    n_customers, shared = np.zeros(len(out)), np.zeros(len(out), dtype=int)
    for i, (dev, cust) in enumerate(zip(out["device_id"], out["customer_id"])):
        bucket = seen_customers.setdefault(dev, set())
        n_customers[i] = len(bucket)
        shared[i] = int(len(bucket - {cust}) > 0)
        bucket.add(cust)
    out["device_n_prev_customers"] = n_customers
    out["device_shared"] = shared
    return out


t0 = time.time()
df_feat = add_behavioural_features(df_clean, CFG)
df_feat = add_device_sharing_features(df_feat, CFG)
print("behavioural + sharing features built in {:.1f}s".format(time.time() - t0))
show(df_feat[["customer_id", CFG.time_col, "amount", "n_txn_prev_1h",
              "n_txn_prev_24h", "amt_sum_prev_24h", "secs_since_prev_txn",
              "amount_vs_cust_mean", "is_first_time_device", "device_shared",
              CFG.target]], 8, "sample of engineered behavioural features")

In [ ]:
# ============================================================================
# 1.5b  Temporal, monetary-context and geography/identity risk features.
# ============================================================================
HIGH_RISK_IPS = set(["ru", "cn", "ng", "ua", "br"])


def add_context_features(frame: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Add calendar, monetary-context and geo/identity mismatch features."""
    out = frame.copy()
    ts = out[cfg.time_col]

    # --- temporal -----------------------------------------------------------
    out["hour"] = ts.dt.hour
    out["day_of_week"] = ts.dt.dayofweek
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["is_night"] = ((out["hour"] < 6) | (out["hour"] >= 23)).astype(int)
    out["day_of_month"] = ts.dt.day
    if "signup_date" in out.columns:
        signup = pd.to_datetime(out["signup_date"], errors="coerce")
        out["days_since_signup"] = (ts - signup).dt.total_seconds() / 86_400.0
        out["is_new_account"] = (out["days_since_signup"] < 30).astype(int)

    # --- monetary context ---------------------------------------------------
    out["log_amount"] = np.log1p(out["amount"])
    if "credit_limit" in out.columns:
        out["amount_to_limit"] = out["amount"] / out["credit_limit"].replace(0, np.nan)
        out["exposure_24h_to_limit"] = (out["amt_sum_prev_24h"]
                                        / out["credit_limit"].replace(0, np.nan))
    out["amount_round_number"] = (out["amount"] % 100 == 0).astype(int)

    # --- geography / identity ----------------------------------------------
    if {"ip_country", "home_country"} <= set(out.columns):
        out["ip_not_home"] = (out["ip_country"] != out["home_country"]).astype(int)
        out["ip_high_risk"] = out["ip_country"].isin(HIGH_RISK_IPS).astype(int)
    if {"merchant_country", "home_country"} <= set(out.columns):
        out["merchant_not_home"] = (out["merchant_country"]
                                    != out["home_country"]).astype(int)
    if {"ip_country", "merchant_country"} <= set(out.columns):
        out["ip_vs_merchant_mismatch"] = (out["ip_country"]
                                          != out["merchant_country"]).astype(int)

    # --- composite, human-readable risk score (a cheap sanity check) --------
    out["heuristic_risk_score"] = (
        out.get("is_night", 0) + out.get("is_first_time_device", 0)
        + out.get("ip_not_home", 0) + out.get("ip_high_risk", 0)
        + out.get("is_proxy_vpn", 0) + out.get("is_burst_1h", 0)
        + (1 - out.get("auth_3ds", 0))
    )
    return out


df_feat = add_context_features(df_feat, CFG)

section("engineered feature sanity check: mean value by class")
engineered = ["log_amount", "amount_vs_cust_mean", "n_txn_prev_1h", "n_txn_prev_24h",
              "secs_since_prev_txn", "is_night", "is_first_time_device",
              "device_shared", "ip_not_home", "ip_high_risk",
              "ip_vs_merchant_mismatch", "is_proxy_vpn", "auth_3ds",
              "amount_to_limit", "heuristic_risk_score"]
engineered = [c for c in engineered if c in df_feat.columns]
cmp_tbl = df_feat.groupby(CFG.target)[engineered].mean().T
cmp_tbl.columns = ["legitimate", "fraud"]
cmp_tbl["lift_fraud_vs_legit"] = (cmp_tbl["fraud"]
                                  / cmp_tbl["legitimate"].replace(0, np.nan))
show(cmp_tbl.sort_values("lift_fraud_vs_legit", ascending=False),
     len(cmp_tbl), "class separation achieved by the engineered features")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))
for ax, col in zip(axes, ["heuristic_risk_score", "n_txn_prev_1h",
                          "amount_vs_cust_mean"]):
    for label, colour, name in [(0, "#4C78A8", "legit"), (1, "#E45756", "fraud")]:
        vals = df_feat.loc[df_feat[CFG.target] == label, col].replace(
            [np.inf, -np.inf], np.nan).dropna()
        if col == "amount_vs_cust_mean":
            vals = vals.clip(upper=vals.quantile(0.99))
        ax.hist(vals, bins=30, density=True, alpha=0.6, color=colour, label=name)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 1.6 Feature selection, chronological split and leakage control

**Why not `train_test_split(shuffle=True)`?** Fraud is a *forecasting* problem: in production the
model is trained on the past and scored on the future. A random split lets the model see
transactions from the same attack burst on both sides of the split, which inflates every metric.
We therefore split **chronologically** into train (60%) / validation (20%) / test (20%).

**Columns deliberately dropped from the feature matrix**

| Dropped | Reason |
|---|---|
| `txn_id`, `customer_id`, `device_id` | identifiers — memorising them cannot generalise (they are still used to *derive* features) |
| `timestamp`, `signup_date` | raw datetimes; the usable content is already encoded as calendar + tenure features |
| `gender` | **protected attribute** — excluded from the model, retained only for the fairness audit in Section 6 |
| `region`, `home_country` | kept, but audited for proxy discrimination in Section 6 |

In [ ]:
# ============================================================================
# 1.6a  Declare the feature contract explicitly.
# ============================================================================
EXCLUDED_IDENTIFIERS = ["txn_id", "customer_id", "device_id"]
EXCLUDED_RAW_TIME = [CFG.time_col, "signup_date"]
EXCLUDED_PROTECTED = ["gender"]                      # fairness: not a model input
SENSITIVE_FOR_AUDIT = ["gender", "age", "region", "account_type", "home_country"]

CANDIDATE_NUMERIC = [
    "amount", "log_amount", "amount_to_limit", "exposure_24h_to_limit",
    "amount_round_number", "amount_vs_cust_mean", "cust_prev_mean_amount",
    "cust_txn_seq", "n_txn_prev_1h", "n_txn_prev_24h", "n_txn_prev_7d",
    "amt_sum_prev_1h", "amt_sum_prev_24h", "amt_sum_prev_7d",
    "secs_since_prev_txn", "hour", "day_of_week", "day_of_month", "is_weekend",
    "is_night", "days_since_signup", "is_new_account", "age", "credit_limit",
    "device_age_days", "os_major", "device_n_prev_customers", "device_shared",
    "is_new_device", "is_first_time_device", "ip_country_changed",
    "ip_not_home", "ip_high_risk", "merchant_not_home",
    "ip_vs_merchant_mismatch", "is_proxy_vpn", "is_rooted", "card_present",
    "auth_3ds", "is_burst_1h", "heuristic_risk_score",
]
CANDIDATE_CATEGORICAL = [
    "merchant_category", "channel", "ip_country", "merchant_country",
    "home_country", "region", "account_type", "kyc_level", "device_type",
    "screen_class",
]

NUMERIC_FEATURES = [c for c in CANDIDATE_NUMERIC if c in df_feat.columns]
CATEGORICAL_FEATURES = [c for c in CANDIDATE_CATEGORICAL if c in df_feat.columns]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

model_data = df_feat.sort_values(CFG.time_col).reset_index(drop=True)
model_data[NUMERIC_FEATURES] = (model_data[NUMERIC_FEATURES]
                                .replace([np.inf, -np.inf], np.nan))

section("feature contract")
print("numeric features     : {:>3}".format(len(NUMERIC_FEATURES)))
print("categorical features : {:>3}".format(len(CATEGORICAL_FEATURES)))
print("total model inputs   : {:>3}".format(len(FEATURES)))
dropped = [c for c in model_data.columns
           if c not in FEATURES + [CFG.target] + EXCLUDED_IDENTIFIERS
           + EXCLUDED_RAW_TIME]
print("\ndropped identifiers  :", EXCLUDED_IDENTIFIERS)
print("dropped raw datetimes:", EXCLUDED_RAW_TIME)
print("dropped protected    :", EXCLUDED_PROTECTED)
print("other unused columns :", dropped if dropped else "none")
print("\nnumeric     :", NUMERIC_FEATURES)
print("\ncategorical :", CATEGORICAL_FEATURES)

In [ ]:
# ============================================================================
# 1.6b  Chronological train / validation / test split.
# ============================================================================
def chronological_split(frame: pd.DataFrame, cfg: Config
                        ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split a time-sorted frame into past (train), recent (val), future (test)."""
    n = len(frame)
    i_train = int(n * cfg.train_frac)
    i_val = int(n * (cfg.train_frac + cfg.val_frac))
    return (frame.iloc[:i_train].copy(), frame.iloc[i_train:i_val].copy(),
            frame.iloc[i_val:].copy())


train_df, val_df, test_df = chronological_split(model_data, CFG)

# --- leakage-safe frequency encoding: fitted on TRAIN, applied everywhere ---
def add_frequency_encodings(train: pd.DataFrame, others: Sequence[pd.DataFrame],
                            cols: Sequence[str] = ("customer_id", "device_id")
                            ) -> None:
    """In-place target-free frequency encoding for high-cardinality keys.

    Frequencies come from the training period only; entities never seen during
    training get 0.0, which is itself meaningful ("brand-new entity").
    """
    for col in cols:
        if col not in train.columns:
            continue
        freq = train[col].value_counts(normalize=True)
        new_col = col + "_freq_train"
        train[new_col] = train[col].map(freq).fillna(0.0)
        for frame in others:
            frame[new_col] = frame[col].map(freq).fillna(0.0)
        if new_col not in NUMERIC_FEATURES:
            NUMERIC_FEATURES.append(new_col)


add_frequency_encodings(train_df, [val_df, test_df])
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train, y_train = train_df[FEATURES], train_df[CFG.target].astype(int)
X_val, y_val = val_df[FEATURES], val_df[CFG.target].astype(int)
X_test, y_test = test_df[FEATURES], test_df[CFG.target].astype(int)

amt_val = val_df["amount"].to_numpy()
amt_test = test_df["amount"].to_numpy()

section("split summary (chronological, no shuffling)")
summary = pd.DataFrame([
    {"split": name, "rows": len(part), "from": part[CFG.time_col].min(),
     "to": part[CFG.time_col].max(), "frauds": int(part[CFG.target].sum()),
     "fraud_rate_%": round(100 * part[CFG.target].mean(), 3)}
    for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]
])
print(summary.to_string(index=False))
print("\nfeature matrix: X_train={}  X_val={}  X_test={}".format(
    X_train.shape, X_val.shape, X_test.shape))
assert train_df[CFG.time_col].max() <= val_df[CFG.time_col].min(), "temporal overlap!"
assert val_df[CFG.time_col].max() <= test_df[CFG.time_col].min(), "temporal overlap!"
print("temporal ordering asserted: train < validation < test  [OK]")

> **Section 1 summary.** Three tables were joined with an audited left join; the feed's real
> defects (300 replayed rows, missing categoricals, impossible amounts, inconsistent casing) were
> repaired with a logged, count-by-count procedure; and ~45 features across four hypothesis-driven
> families were engineered from past-only aggregates. The engineered columns show far stronger
> class separation than any raw column, which is where most of this model's performance comes
> from. The split is chronological, the frequency encodings were fitted on the training period
> only, and imputation/scaling live inside the model pipeline — so the evaluation in Sections 3-4
> is leakage-free.

<a name="sec2"></a>
# 2. Model Selection  *(3 marks)*

### 2.1 What we are selecting between, and why

| Candidate | Why it is in the shortlist | Known weakness here |
|---|---|---|
| **Dummy (prior)** | mandatory floor: any model must beat "flag nothing" | no discrimination at all |
| **Logistic Regression** | fast, monotone, coefficients are directly explainable to a regulator | cannot express interactions (`night` x `new device` x `amount`) without manual crosses |
| **Decision Tree** | human-readable rules that fraud analysts can adopt as policy | high variance, poor probability estimates |
| **Random Forest** | strong bagged non-linear baseline, robust to outliers/scaling | large model, slower scoring, weaker on very rare classes than boosting |
| **Hist Gradient Boosting** | state of the art on tabular data, native missing-value handling, fast | needs tuning, less interpretable (addressed in Section 5) |
| **XGBoost** *(if installed)* | boosting with `scale_pos_weight`, industry default for fraud | extra dependency |
| **LogReg + SMOTE** *(if installed)* | tests whether **synthetic oversampling** beats cost-sensitive weighting | SMOTE interpolates between rare frauds and can manufacture unrealistic points |

Three design decisions are being tested at the same time as the algorithm:

1. **Imbalance strategy** — `class_weight='balanced'` (cost-sensitive learning) vs SMOTE
   resampling vs nothing-plus-threshold-tuning.
2. **Validation scheme** — `TimeSeriesSplit` on the training period only. Each fold trains on the
   past and validates on the immediate future, which mirrors deployment; the validation and test
   periods are never touched during selection.
3. **Selection metric** — **PR-AUC (average precision)**. With a ~1.5% positive rate, ROC-AUC is
   dominated by the huge negative class and looks flattering; PR-AUC responds to what the fraud
   team actually feels — precision at the top of the alert queue. It is reported alongside for
   reference.

In [ ]:
# ============================================================================
# 2.1  Preprocessing pipeline + candidate model zoo.
# ============================================================================
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


def build_preprocessor(numeric: Sequence[str],
                       categorical: Sequence[str]) -> ColumnTransformer:
    """Impute + scale numerics, impute + one-hot encode categoricals.

    Every statistic (medians, modes, scaling parameters, category vocabulary)
    is learned inside the pipeline, therefore re-learned on the training part
    of every CV fold. This is what makes the cross-validation honest.
    `add_indicator=True` keeps 'was missing' as an explicit signal - a missing
    device fingerprint is suspicious in itself.
    """
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("encode", make_ohe()),
    ])
    return ColumnTransformer(
        [("num", numeric_pipe, list(numeric)),
         ("cat", categorical_pipe, list(categorical))],
        remainder="drop",
    )


def make_pipeline(model, sampler=None) -> Pipeline:
    """Wrap a classifier (optionally with a resampler) behind the preprocessor."""
    steps = [("prep", build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES))]
    if sampler is not None and ImbPipeline is not None:
        steps.append(("resample", sampler))
        steps.append(("model", model))
        return ImbPipeline(steps)
    steps.append(("model", model))
    return Pipeline(steps)


# imbalance ratio used for cost-sensitive boosting
POS_WEIGHT = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print("scale_pos_weight (negatives per positive in train): {:.1f}".format(POS_WEIGHT))


def candidate_models() -> Dict[str, Pipeline]:
    """Fresh, unfitted candidate pipelines (defaults - tuning happens later)."""
    zoo: Dict[str, Pipeline] = {
        "Dummy (prior)": make_pipeline(
            DummyClassifier(strategy="prior")),
        "Logistic Regression": make_pipeline(
            LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=CFG.random_state)),
        "Decision Tree": make_pipeline(
            DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                   class_weight="balanced",
                                   random_state=CFG.random_state)),
        "Random Forest": make_pipeline(
            RandomForestClassifier(n_estimators=250, min_samples_leaf=3,
                                   max_features="sqrt", n_jobs=-1,
                                   class_weight="balanced_subsample",
                                   random_state=CFG.random_state)),
        "Hist Gradient Boosting": make_pipeline(
            safe_estimator(HistGradientBoostingClassifier, max_iter=300,
                           learning_rate=0.08, max_leaf_nodes=31,
                           min_samples_leaf=25, l2_regularization=1.0,
                           early_stopping=False, class_weight="balanced",
                           random_state=CFG.random_state)),
    }
    if xgb is not None:
        zoo["XGBoost"] = make_pipeline(xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.08, max_depth=5, subsample=0.9,
            colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=2,
            scale_pos_weight=POS_WEIGHT, eval_metric="aucpr", n_jobs=-1,
            tree_method="hist", random_state=CFG.random_state))
    if SMOTE is not None and ImbPipeline is not None:
        zoo["LogReg + SMOTE"] = make_pipeline(
            LogisticRegression(max_iter=2000, random_state=CFG.random_state),
            sampler=SMOTE(sampling_strategy=0.25, k_neighbors=5,
                          random_state=CFG.random_state))
    return zoo


print("candidates:", list(candidate_models()))

In [ ]:
# ============================================================================
# 2.2  Time-series cross-validation on the TRAINING period only.
# ============================================================================
cv = TimeSeriesSplit(n_splits=CFG.cv_splits)
SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc",
           "recall": "recall", "precision": "precision"}

print("TimeSeriesSplit folds (train rows -> validation rows):")
for k, (tr, va) in enumerate(cv.split(X_train), 1):
    print("  fold {}: {:>6,} -> {:>6,}   frauds in fold-val: {:>3}".format(
        k, len(tr), len(va), int(y_train.iloc[va].sum())))

rows = []
for name, pipe in candidate_models().items():
    t0 = time.time()
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING,
                            n_jobs=1, error_score="raise")
    rows.append({
        "model": name,
        "pr_auc_mean": scores["test_pr_auc"].mean(),
        "pr_auc_std": scores["test_pr_auc"].std(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "recall@0.5": scores["test_recall"].mean(),
        "precision@0.5": scores["test_precision"].mean(),
        "fit_seconds": scores["fit_time"].mean(),
    })
    print("  {:<24} PR-AUC={:.4f} (+/-{:.4f})  ROC-AUC={:.4f}  [{:.1f}s]".format(
        name, rows[-1]["pr_auc_mean"], rows[-1]["pr_auc_std"],
        rows[-1]["roc_auc_mean"], time.time() - t0))

cv_results = (pd.DataFrame(rows).sort_values("pr_auc_mean", ascending=False)
              .reset_index(drop=True))
section("cross-validated model comparison (selection metric: PR-AUC)")
show(cv_results, len(cv_results), "candidate ranking")

In [ ]:
# ============================================================================
# 2.3  Visual comparison + an auto-generated (never fabricated) conclusion.
# ============================================================================
plot_df = cv_results.sort_values("pr_auc_mean")
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))

ax = axes[0]
ax.barh(plot_df["model"], plot_df["pr_auc_mean"],
        xerr=plot_df["pr_auc_std"], color="#4C78A8", alpha=0.9)
ax.axvline(float(y_train.mean()), ls="--", c="#E45756",
           label="random baseline = base rate")
ax.set_title("Cross-validated PR-AUC (higher is better)")
ax.set_xlabel("average precision")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(cv_results["fit_seconds"], cv_results["pr_auc_mean"],
           s=70, color="#E45756")
for _, r in cv_results.iterrows():
    ax.annotate(r["model"], (r["fit_seconds"], r["pr_auc_mean"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log")
ax.set_title("Accuracy vs training cost trade-off")
ax.set_xlabel("mean fit time per fold (s, log)")
ax.set_ylabel("PR-AUC")
plt.tight_layout()
plt.show()

best_row = cv_results.iloc[0]
runner_row = cv_results.iloc[1]
dummy_row = cv_results[cv_results["model"] == "Dummy (prior)"].iloc[0]
BEST_MODEL_NAME = str(best_row["model"])
RUNNER_UP_NAME = str(runner_row["model"])

section("selection decision (generated from the table above)")
print("Winner            : {}  PR-AUC {:.4f} (+/- {:.4f})".format(
    BEST_MODEL_NAME, best_row["pr_auc_mean"], best_row["pr_auc_std"]))
print("Runner-up         : {}  PR-AUC {:.4f}".format(
    RUNNER_UP_NAME, runner_row["pr_auc_mean"]))
print("Trivial baseline  : {}  PR-AUC {:.4f}  (= the fraud base rate)".format(
    dummy_row["model"], dummy_row["pr_auc_mean"]))
print("Lift over baseline: {:.1f}x".format(
    best_row["pr_auc_mean"] / max(dummy_row["pr_auc_mean"], 1e-9)))
lr_row = cv_results[cv_results["model"] == "Logistic Regression"].iloc[0]
print("Non-linearity pays: {:+.1f}% PR-AUC vs Logistic Regression".format(
    100 * (best_row["pr_auc_mean"] / max(lr_row["pr_auc_mean"], 1e-9) - 1)))
if (cv_results["model"] == "LogReg + SMOTE").any():
    smote_row = cv_results[cv_results["model"] == "LogReg + SMOTE"].iloc[0]
    print("SMOTE vs class_weight on the same learner: {:.4f} vs {:.4f} PR-AUC"
          " -> {}".format(smote_row["pr_auc_mean"], lr_row["pr_auc_mean"],
                          "SMOTE helps" if smote_row["pr_auc_mean"]
                          > lr_row["pr_auc_mean"] else
                          "cost-sensitive weighting is preferred (and cheaper)"))
print("\nNote how ROC-AUC compresses the differences ({:.3f} vs {:.3f} for the top two)"
      "\nwhile PR-AUC separates them - the reason PR-AUC is the selection metric."
      .format(best_row["roc_auc_mean"], runner_row["roc_auc_mean"]))
print("\nCARRIED FORWARD: '{}' and '{}' go to Sections 3-4."
      .format(BEST_MODEL_NAME, RUNNER_UP_NAME))

<a name="sec3"></a>
# 3. Performance Measurement  *(3 marks)*

### 3.1 Choosing metrics that match the business problem

With a ~1.5% fraud rate, **accuracy is actively misleading** — a model that never fires already
scores ~98.5%. The metric suite below is built around how the alerts are actually consumed:

| Metric | Question it answers | Why it matters here |
|---|---|---|
| **PR-AUC / average precision** | across all thresholds, how pure is the alert queue? | primary threshold-free metric under imbalance |
| **ROC-AUC** | can the model rank a random fraud above a random legit? | comparable across datasets, but optimistic when positives are rare |
| **Recall (fraud capture rate)** | what share of fraud do we catch? | the loss-prevention headline |
| **Precision** | what share of alerts are real? | analyst trust and cost |
| **F1 / F2** | balance of the two (F2 favours recall) | fraud usually prefers recall, so F2 is reported |
| **Recall @ precision >= 30%** | recall at a precision the operations team accepts | the deployable operating point |
| **Precision @ 1% alert rate** | quality of the queue an analyst team can actually clear | capacity-constrained reality |
| **Brier score / calibration** | are the probabilities *honest*? | required for cost-based decisions and risk-based step-up auth |
| **Expected cost (USD)** | money | the metric management signs off on |

**Confusion-matrix vocabulary in this domain:** a *false negative* is fraud we paid for, a *false
positive* is a blocked genuine customer (analyst time **plus** goodwill damage). They are not
equally expensive, so the decision threshold is chosen by cost in 3.4 — not left at 0.5.

In [ ]:
# ============================================================================
# 3.1  Metric toolkit.
# ============================================================================
from sklearn.calibration import calibration_curve
from sklearn.metrics import (auc, average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix, f1_score,
                             fbeta_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)


def recall_at_precision(y_true, scores, min_precision: float = 0.30
                        ) -> Tuple[float, float]:
    """Best achievable recall subject to precision >= `min_precision`.

    Returns (recall, threshold); (0, 1) when the constraint is unreachable.
    """
    precision, recall, thresh = precision_recall_curve(y_true, scores)
    ok = precision[:-1] >= min_precision
    if not ok.any():
        return 0.0, 1.0
    idx = int(np.argmax(np.where(ok, recall[:-1], -1)))
    return float(recall[idx]), float(thresh[idx])


def queue_metrics(y_true, scores, alert_rate: float) -> Dict[str, float]:
    """Precision/recall of the top `alert_rate` share of scores (analyst queue)."""
    y_true = np.asarray(y_true)
    k = max(1, int(round(len(scores) * alert_rate)))
    top = np.argsort(-np.asarray(scores))[:k]
    caught = float(y_true[top].sum())
    return {"alerts": k,
            "precision": caught / k,
            "recall": caught / max(y_true.sum(), 1)}


def threshold_free_metrics(y_true, scores) -> Dict[str, float]:
    """Metrics that do not depend on a decision threshold."""
    rap, rap_t = recall_at_precision(y_true, scores, 0.30)
    q = queue_metrics(y_true, scores, CFG.analyst_capacity)
    return {"pr_auc": average_precision_score(y_true, scores),
            "roc_auc": roc_auc_score(y_true, scores),
            "brier": brier_score_loss(y_true, scores),
            "recall@prec>=0.30": rap,
            "thr@prec>=0.30": rap_t,
            "precision@1%alerts": q["precision"],
            "recall@1%alerts": q["recall"]}


def threshold_metrics(y_true, scores, threshold: float) -> Dict[str, float]:
    """Point metrics for one decision threshold."""
    y_pred = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"threshold": threshold, "tp": int(tp), "fp": int(fp), "fn": int(fn),
            "tn": int(tn),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
            "alert_rate": float(y_pred.mean()),
            "accuracy_(misleading)": float((y_pred == np.asarray(y_true)).mean())}


def expected_cost(y_true, y_pred, amounts, cfg: Config) -> Dict[str, float]:
    """Translate a confusion matrix into money.

    Cost model (documented assumptions, all configurable in `Config`):
      * false negative -> the fraud is paid, minus a `recovery_rate` chargeback
      * false positive -> analyst review cost + customer-friction/goodwill cost
      * true positive  -> analyst review cost, but the loss is avoided
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    amounts = np.asarray(amounts, dtype=float)
    fn = (y_true == 1) & (y_pred == 0)
    fp = (y_true == 0) & (y_pred == 1)
    tp = (y_true == 1) & (y_pred == 1)
    loss_fn = amounts[fn].sum() * (1.0 - cfg.recovery_rate)
    loss_fp = fp.sum() * (cfg.review_cost + cfg.friction_cost)
    loss_tp = tp.sum() * cfg.review_cost
    total = loss_fn + loss_fp + loss_tp
    do_nothing = amounts[y_true == 1].sum() * (1.0 - cfg.recovery_rate)
    return {"fraud_loss": loss_fn, "review+friction_cost": loss_fp + loss_tp,
            "total_cost": total, "cost_if_no_model": do_nothing,
            "net_saving": do_nothing - total}


print("metric toolkit ready")

In [ ]:
# ============================================================================
# 3.2  Fit the two selected candidates on TRAIN and score the VALIDATION set.
#      The test set stays sealed until Section 4.4.
# ============================================================================
zoo = candidate_models()
selected = {name: zoo[name] for name in [BEST_MODEL_NAME, RUNNER_UP_NAME]}

fitted: Dict[str, Pipeline] = {}
val_scores: Dict[str, np.ndarray] = {}
rows = []
for name, pipe in selected.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    s = pipe.predict_proba(X_val)[:, 1]
    fitted[name], val_scores[name] = pipe, s
    row = {"model": name}
    row.update(threshold_free_metrics(y_val, s))
    row["fit_seconds"] = time.time() - t0
    rows.append(row)

val_table = pd.DataFrame(rows).set_index("model")
section("validation performance (threshold-free)")
show(val_table.T, len(val_table.T), "threshold-independent metrics")

print("\nThe naive 0.5 threshold, for contrast:")
naive = pd.DataFrame([dict({"model": n},
                           **threshold_metrics(y_val, s, 0.5))
                      for n, s in val_scores.items()]).set_index("model")
print(naive.T.to_string())
print("\n^ note the 'accuracy_(misleading)' row: ~98-99% for every model,"
      " including ones\n  with poor precision. This is the imbalance trap"
      " that Section 3.1 warns about.")

In [ ]:
# ============================================================================
# 3.3  Curves: precision-recall, ROC and reliability (calibration).
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
colours = ["#4C78A8", "#E45756", "#72B7B2"]

# --- precision-recall -------------------------------------------------------
ax = axes[0]
for (name, s), c in zip(val_scores.items(), colours):
    p, r, _ = precision_recall_curve(y_val, s)
    ax.plot(r, p, color=c, label="{} (AP={:.3f})".format(
        name, average_precision_score(y_val, s)))
ax.axhline(y_val.mean(), ls="--", c="grey",
           label="no-skill = {:.3f}".format(y_val.mean()))
ax.axhline(0.30, ls=":", c="#54A24B", label="operations floor (P=0.30)")
ax.set_xlabel("recall (fraud caught)")
ax.set_ylabel("precision (alert purity)")
ax.set_title("Precision-Recall (the metric that matters)")
ax.legend(fontsize=8, loc="upper right")

# --- ROC --------------------------------------------------------------------
ax = axes[1]
for (name, s), c in zip(val_scores.items(), colours):
    fpr, tpr, _ = roc_curve(y_val, s)
    ax.plot(fpr, tpr, color=c, label="{} (AUC={:.3f})".format(
        name, roc_auc_score(y_val, s)))
ax.plot([0, 1], [0, 1], ls="--", c="grey", label="chance")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("ROC - flattering under imbalance")
ax.legend(fontsize=8, loc="lower right")

# --- calibration ------------------------------------------------------------
ax = axes[2]
for (name, s), c in zip(val_scores.items(), colours):
    frac_pos, mean_pred = calibration_curve(y_val, s, n_bins=10,
                                            strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", color=c,
            label="{} (Brier={:.4f})".format(name, brier_score_loss(y_val, s)))
ax.plot([0, 1], [0, 1], ls="--", c="grey", label="perfectly calibrated")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed fraud rate")
ax.set_title("Reliability curve")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Reading the calibration plot: `class_weight='balanced'` and SMOTE both"
      " inflate\npredicted probabilities (the curve sits below the diagonal)."
      " Ranking is unaffected,\nbut the raw numbers are not true probabilities"
      " - so cost calculations must use a\nthreshold tuned on data (3.4) or a"
      " calibrated wrapper, never a naive 0.5 cut.")

In [ ]:
# ============================================================================
# 3.4  Threshold selection = a BUSINESS decision, made on validation data.
# ============================================================================
def sweep_thresholds(y_true, scores, amounts, cfg: Config,
                     n: int = 220) -> pd.DataFrame:
    """Evaluate operating points across the score range."""
    qs = np.unique(np.quantile(scores, np.linspace(0.50, 0.9999, n)))
    out = []
    for t in qs:
        m = threshold_metrics(y_true, scores, t)
        m.update(expected_cost(y_true, (np.asarray(scores) >= t).astype(int),
                               amounts, cfg))
        out.append(m)
    return pd.DataFrame(out)


PRIMARY = BEST_MODEL_NAME
sweep = sweep_thresholds(y_val, val_scores[PRIMARY], amt_val, CFG)

# candidate operating policies -----------------------------------------------
i_cost = int(sweep["total_cost"].idxmin())
i_f2 = int(sweep["f2"].idxmax())
feasible = sweep[sweep["precision"] >= 0.30]
i_prec = int(feasible["recall"].idxmax()) if len(feasible) else i_cost
capacity = sweep[sweep["alert_rate"] <= CFG.analyst_capacity]
i_cap = int(capacity["recall"].idxmax()) if len(capacity) else i_cost

policies = pd.DataFrame([
    dict({"policy": "min expected cost (chosen)"}, **sweep.loc[i_cost].to_dict()),
    dict({"policy": "max F2"}, **sweep.loc[i_f2].to_dict()),
    dict({"policy": "max recall s.t. precision>=0.30"}, **sweep.loc[i_prec].to_dict()),
    dict({"policy": "max recall within 1% alert capacity"}, **sweep.loc[i_cap].to_dict()),
    dict({"policy": "naive 0.5"}, **dict(threshold_metrics(y_val, val_scores[PRIMARY], 0.5),
                                         **expected_cost(y_val, (val_scores[PRIMARY] >= 0.5).astype(int),
                                                         amt_val, CFG))),
]).set_index("policy")

section("operating-point comparison on validation ({})".format(PRIMARY))
cols = ["threshold", "precision", "recall", "f1", "f2", "alert_rate", "tp", "fp",
        "fn", "fraud_loss", "review+friction_cost", "total_cost", "net_saving"]
show(policies[cols], len(policies), "candidate policies")

THRESHOLD = float(sweep.loc[i_cost, "threshold"])
print("\nCHOSEN THRESHOLD = {:.4f}  (minimises expected cost on validation)".format(
    THRESHOLD))
print("  cost with no model : {}".format(money(sweep.loc[i_cost, "cost_if_no_model"])))
print("  cost with model    : {}".format(money(sweep.loc[i_cost, "total_cost"])))
print("  net saving         : {}  ({:.1f}% of avoidable loss)".format(
    money(sweep.loc[i_cost, "net_saving"]),
    100 * sweep.loc[i_cost, "net_saving"] / max(sweep.loc[i_cost, "cost_if_no_model"], 1)))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
ax = axes[0]
ax.plot(sweep["threshold"], sweep["total_cost"], color="#E45756", label="total cost")
ax.plot(sweep["threshold"], sweep["fraud_loss"], ls="--", color="#4C78A8",
        label="fraud losses (FN)")
ax.plot(sweep["threshold"], sweep["review+friction_cost"], ls="--", color="#54A24B",
        label="review + friction (FP/TP)")
ax.axvline(THRESHOLD, color="black", ls=":", label="chosen threshold")
ax.set_xscale("log")
ax.set_xlabel("decision threshold (log)")
ax.set_ylabel("expected cost (USD)")
ax.set_title("Cost curve: the optimum is NOT 0.5")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(sweep["threshold"], sweep["precision"], label="precision", color="#4C78A8")
ax.plot(sweep["threshold"], sweep["recall"], label="recall", color="#E45756")
ax.plot(sweep["threshold"], sweep["f2"], label="F2", color="#54A24B")
ax.plot(sweep["threshold"], sweep["alert_rate"], label="alert rate", color="#B279A2")
ax.axvline(THRESHOLD, color="black", ls=":")
ax.axhline(CFG.analyst_capacity, color="grey", ls="--", lw=0.8,
           label="analyst capacity")
ax.set_xscale("log")
ax.set_xlabel("decision threshold (log)")
ax.set_title("Precision / recall / workload trade-off")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 3.5  Confusion matrix and per-class report at the chosen threshold.
# ============================================================================
y_val_pred = (val_scores[PRIMARY] >= THRESHOLD).astype(int)
cm = confusion_matrix(y_val, y_val_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5.2, 4.2))
im = ax.imshow(cm, cmap="Blues")
labels = [["TN\ncorrectly allowed", "FP\ngenuine blocked"],
          ["FN\nfraud missed", "TP\nfraud caught"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i, "{}\n{:,}".format(labels[i][j], cm[i, j]),
                ha="center", va="center", fontsize=10,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["predicted legit", "predicted fraud"])
ax.set_yticks([0, 1], ["actual legit", "actual fraud"])
ax.set_title("{} @ threshold {:.4f} (validation)".format(PRIMARY, THRESHOLD))
ax.grid(False)
plt.tight_layout()
plt.show()

print(classification_report(y_val, y_val_pred, digits=4, zero_division=0,
                            target_names=["legitimate", "fraud"]))
tn, fp, fn, tp = cm.ravel()
print("Operational read-out")
print("  fraud caught      : {:,} of {:,}  ({:.1%} recall)".format(
    tp, tp + fn, tp / max(tp + fn, 1)))
print("  alerts raised     : {:,}  ({:.2f}% of traffic)".format(
    tp + fp, 100 * (tp + fp) / len(y_val)))
print("  alert purity      : {:.1%} precision -> analysts review"
      " ~{:.1f} cases per catch".format(tp / max(tp + fp, 1),
                                        (tp + fp) / max(tp, 1)))
print("  value not blocked : {} of fraud value slipped through".format(
    money(amt_val[(y_val.to_numpy() == 1) & (y_val_pred == 0)].sum())))
print("  value protected   : {}".format(
    money(amt_val[(y_val.to_numpy() == 1) & (y_val_pred == 1)].sum())))

<a name="sec4"></a>
# 4. Hyperparameter Tuning  *(5 marks)*

### 4.1 Strategy

| Decision | Choice | Rationale |
|---|---|---|
| **Search algorithm** | `RandomizedSearchCV`, 30 draws | with 6-8 interacting hyper-parameters a full grid is combinatorially wasteful; random search finds equally good regions in a fraction of the budget (Bergstra & Bengio, 2012) |
| **Validation inside the search** | `TimeSeriesSplit(3)` **on the training period only** | keeps the temporal ordering *and* keeps validation/test unseen, so the tuned score is not optimistically biased |
| **Optimisation objective** | `average_precision` (PR-AUC) | same metric the model was selected on; optimising accuracy or ROC-AUC would tune for the wrong thing |
| **Search space** | capacity (`max_leaf_nodes`, `max_depth`, `n_estimators`) + regularisation (`l2`, `min_samples_leaf`, `subsample`) + `learning_rate` | jointly controls the bias-variance trade-off, which is the real reason to tune |
| **What is *not* tuned here** | the decision threshold | it is a business choice re-derived from the cost model *after* tuning (4.3), not a model hyper-parameter |
| **Overfitting guard** | tuned on train-only folds → checked on validation → reported once on test | a single, final touch of the test set |

Two families are tuned so the comparison is fair: the winning non-linear model **and** the
interpretable logistic-regression challenger.

In [ ]:
# ============================================================================
# 4.1  Search spaces (kept as explicit lists so the run is fully reproducible).
# ============================================================================
def param_space(name: str) -> Dict[str, List[Any]]:
    """Randomised-search distribution for a given candidate name."""
    spaces: Dict[str, Dict[str, List[Any]]] = {
        "Hist Gradient Boosting": {
            "model__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.12, 0.20],
            "model__max_iter": [200, 300, 450, 600],
            "model__max_leaf_nodes": [15, 31, 63, 127],
            "model__max_depth": [None, 4, 6, 8, 12],
            "model__min_samples_leaf": [10, 20, 40, 80, 150],
            "model__l2_regularization": [0.0, 0.1, 0.5, 1.0, 5.0, 10.0],
            "model__max_bins": [128, 255],
        },
        "XGBoost": {
            "model__n_estimators": [200, 350, 500, 700],
            "model__learning_rate": [0.02, 0.05, 0.08, 0.12, 0.2],
            "model__max_depth": [3, 4, 5, 6, 8],
            "model__min_child_weight": [1, 2, 5, 10],
            "model__subsample": [0.7, 0.8, 0.9, 1.0],
            "model__colsample_bytree": [0.6, 0.8, 1.0],
            "model__reg_lambda": [0.5, 1.0, 5.0, 20.0],
            "model__gamma": [0.0, 0.5, 2.0],
            "model__scale_pos_weight": [1.0, np.sqrt(POS_WEIGHT), POS_WEIGHT],
        },
        "Random Forest": {
            "model__n_estimators": [200, 400, 600],
            "model__max_depth": [None, 8, 14, 22],
            "model__min_samples_leaf": [1, 2, 4, 8, 16],
            "model__max_features": ["sqrt", "log2", 0.3, 0.5],
            "model__class_weight": ["balanced", "balanced_subsample", None],
        },
        "Decision Tree": {
            "model__max_depth": [3, 4, 6, 8, 12, None],
            "model__min_samples_leaf": [10, 25, 50, 100, 250],
            "model__criterion": ["gini", "entropy"],
            "model__ccp_alpha": [0.0, 1e-5, 1e-4, 1e-3],
            "model__class_weight": ["balanced", None],
        },
        "Logistic Regression": {
            "model__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
            "model__class_weight": ["balanced", None],
            "model__solver": ["lbfgs", "liblinear"],
        },
        "LogReg + SMOTE": {
            "model__C": [0.01, 0.1, 1.0, 10.0],
            "resample__sampling_strategy": [0.1, 0.25, 0.5, 1.0],
            "resample__k_neighbors": [3, 5, 10],
        },
    }
    return spaces.get(name, {"model__C": [0.1, 1.0, 10.0]})


def tune(name: str, n_iter: int, cv_splits: int = 3,
         n_jobs: int = 2) -> RandomizedSearchCV:
    """Run a randomised PR-AUC search for one candidate on the training data."""
    search = RandomizedSearchCV(
        estimator=candidate_models()[name],
        param_distributions=param_space(name),
        n_iter=n_iter,
        scoring="average_precision",
        cv=TimeSeriesSplit(n_splits=cv_splits),
        random_state=CFG.random_state,
        n_jobs=n_jobs,
        refit=True,
        verbose=0,
        error_score=-np.inf,
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    print("  {:<24} best CV PR-AUC={:.4f}   ({} fits in {:.0f}s)".format(
        name, search.best_score_, n_iter * cv_splits, time.time() - t0))
    return search


TUNE_TARGETS = [BEST_MODEL_NAME]
if "Logistic Regression" not in TUNE_TARGETS:
    TUNE_TARGETS.append("Logistic Regression")

section("randomised search ({} draws x 3 temporal folds per model)".format(
    CFG.search_iterations))
searches: Dict[str, RandomizedSearchCV] = {}
for name in TUNE_TARGETS:
    n_iter = CFG.search_iterations if name != "Logistic Regression" else 12
    searches[name] = tune(name, n_iter=n_iter)

In [ ]:
# ============================================================================
# 4.2  What the search learned: best parameters and the top of the trace.
# ============================================================================
for name, search in searches.items():
    section("best hyper-parameters: {}".format(name))
    for param, value in sorted(search.best_params_.items()):
        print("  {:<38} {}".format(param.replace("model__", ""), value))
    print("  {:<38} {:.4f}".format("--> mean CV PR-AUC", search.best_score_))

    trace = pd.DataFrame(search.cv_results_)
    keep = ["mean_test_score", "std_test_score", "rank_test_score",
            "mean_fit_time"] + [c for c in trace.columns if c.startswith("param_")]
    show(trace[keep].sort_values("rank_test_score").head(8), 8,
         "top 8 configurations for {}".format(name))

# stability of the search: how much did the hyper-parameters actually matter?
fig, axes = plt.subplots(1, len(searches), figsize=(6.6 * len(searches), 3.9),
                         squeeze=False)
for ax, (name, search) in zip(axes[0], searches.items()):
    scores = pd.DataFrame(search.cv_results_)["mean_test_score"].astype(float)
    order = np.arange(1, len(scores) + 1)
    ax.plot(order, scores.sort_values(ascending=False).to_numpy(), marker="o",
            ms=3, color="#4C78A8")
    ax.axhline(scores.max(), ls=":", c="#E45756",
               label="best = {:.4f}".format(scores.max()))
    ax.axhline(scores.median(), ls="--", c="grey",
               label="median = {:.4f}".format(scores.median()))
    ax.set_title("{}: spread of sampled configs".format(name))
    ax.set_xlabel("configuration (sorted)")
    ax.set_ylabel("CV PR-AUC")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("A wide spread means the hyper-parameters matter and the search paid for"
      " itself;\na flat line means the default was already in a good region.")

In [ ]:
# ============================================================================
# 4.3  Did tuning actually help? Default vs tuned, measured on VALIDATION.
# ============================================================================
comparison, tuned_val_scores = [], {}
for name, search in searches.items():
    default_pipe = candidate_models()[name].fit(X_train, y_train)
    s_default = default_pipe.predict_proba(X_val)[:, 1]
    s_tuned = search.best_estimator_.predict_proba(X_val)[:, 1]
    tuned_val_scores[name] = s_tuned
    for variant, s in (("default", s_default), ("tuned", s_tuned)):
        row = {"model": name, "variant": variant}
        row.update(threshold_free_metrics(y_val, s))
        comparison.append(row)

comp = pd.DataFrame(comparison).set_index(["model", "variant"])
section("tuning impact on the validation set")
show(comp[["pr_auc", "roc_auc", "recall@prec>=0.30", "precision@1%alerts",
           "brier"]], len(comp), "default vs tuned")

for name in searches:
    d = comp.loc[(name, "default"), "pr_auc"]
    t = comp.loc[(name, "tuned"), "pr_auc"]
    print("{:<24} PR-AUC {:.4f} -> {:.4f}   ({:+.2f}%)".format(
        name, d, t, 100 * (t / max(d, 1e-9) - 1)))

# --- pick the production model on validation PR-AUC ------------------------
tuned_only = comp.xs("tuned", level="variant")["pr_auc"]
FINAL_NAME = str(tuned_only.idxmax())
FINAL_SEARCH = searches[FINAL_NAME]
print("\nPRODUCTION CANDIDATE: tuned '{}' (validation PR-AUC {:.4f})".format(
    FINAL_NAME, tuned_only.max()))

# --- re-derive the cost-optimal threshold for the tuned model --------------
sweep_tuned = sweep_thresholds(y_val, tuned_val_scores[FINAL_NAME], amt_val, CFG)
THRESHOLD_FINAL = float(sweep_tuned.loc[sweep_tuned["total_cost"].idxmin(),
                                        "threshold"])
print("cost-optimal threshold for the tuned model (from validation): {:.4f}"
      .format(THRESHOLD_FINAL))
print(pd.DataFrame([threshold_metrics(y_val, tuned_val_scores[FINAL_NAME],
                                      THRESHOLD_FINAL)]).T.to_string(header=False))

In [ ]:
# ============================================================================
# 4.4  FINAL, ONE-TIME EVALUATION ON THE SEALED TEST SET.
#
#  The production model is refitted on train + validation (more data, and the
#  most recent weeks are the most relevant ones), reusing the hyper-parameters
#  and the threshold learned earlier. The test period was never used for any
#  decision, so these numbers are the honest generalisation estimate.
# ============================================================================
X_fit = pd.concat([X_train, X_val], axis=0)
y_fit = pd.concat([y_train, y_val], axis=0)

final_model = candidate_models()[FINAL_NAME]
final_model.set_params(**FINAL_SEARCH.best_params_)
t0 = time.time()
final_model.fit(X_fit, y_fit)
print("refit '{}' on {:,} rows (train+validation) in {:.1f}s".format(
    FINAL_NAME, len(X_fit), time.time() - t0))

test_score = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (test_score >= THRESHOLD_FINAL).astype(int)

section("FINAL TEST-SET PERFORMANCE ({} @ threshold {:.4f})".format(
    FINAL_NAME, THRESHOLD_FINAL))
final_free = threshold_free_metrics(y_test, test_score)
final_point = threshold_metrics(y_test, test_score, THRESHOLD_FINAL)
final_cost = expected_cost(y_test, y_test_pred, amt_test, CFG)

final_report = pd.DataFrame({
    "metric": (list(final_free) + list(final_point) + list(final_cost)),
    "value": ([final_free[k] for k in final_free]
              + [final_point[k] for k in final_point]
              + [final_cost[k] for k in final_cost]),
})
print(final_report.to_string(index=False))

print("\n" + classification_report(y_test, y_test_pred, digits=4, zero_division=0,
                                   target_names=["legitimate", "fraud"]))

# --- validation vs test: is the model stable out of time? ------------------
val_final = threshold_free_metrics(y_val, tuned_val_scores[FINAL_NAME])
stability = pd.DataFrame({"validation": val_final, "test": final_free})
stability["delta"] = stability["test"] - stability["validation"]
section("generalisation check (validation vs sealed test)")
print(stability.round(4).to_string())
gap = stability.loc["pr_auc", "delta"]
print("\nPR-AUC changes by {:+.4f} out of time -> {}".format(
    gap, "stable, no material overfitting" if abs(gap) < 0.06
    else "watch this: the drop suggests drift or over-tuning"))

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
ax = axes[0]
p, r, _ = precision_recall_curve(y_test, test_score)
ax.plot(r, p, color="#E45756", label="test (AP={:.3f})".format(final_free["pr_auc"]))
p2, r2, _ = precision_recall_curve(y_val, tuned_val_scores[FINAL_NAME])
ax.plot(r2, p2, color="#4C78A8", ls="--",
        label="validation (AP={:.3f})".format(val_final["pr_auc"]))
ax.axhline(y_test.mean(), ls=":", c="grey", label="no-skill")
ax.set_title("Final model: precision-recall")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.legend(fontsize=8)

ax = axes[1]
cm_test = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
ax.imshow(cm_test, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, "{:,}".format(cm_test[i, j]), ha="center", va="center",
                color="white" if cm_test[i, j] > cm_test.max() / 2 else "black")
ax.set_xticks([0, 1], ["pred legit", "pred fraud"])
ax.set_yticks([0, 1], ["actual legit", "actual fraud"])
ax.set_title("Test confusion matrix")
ax.grid(False)

ax = axes[2]
deciles = pd.DataFrame({"score": test_score, "y": y_test.to_numpy()})
deciles["bin"] = pd.qcut(deciles["score"].rank(method="first"), 10,
                         labels=["D{}".format(i) for i in range(1, 11)])
lift = deciles.groupby("bin", observed=True)["y"].mean() / max(y_test.mean(), 1e-12)
ax.bar([str(i) for i in lift.index], lift.to_numpy(), color="#72B7B2")
ax.axhline(1.0, ls="--", c="grey", label="population average")
ax.set_title("Lift by score decile (D10 = riskiest)")
ax.set_ylabel("x population fraud rate")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

<a name="sec5"></a>
# 5. Extra Features and Considerations  *(3 marks)*

A model that only exists in a notebook prevents zero fraud. This section covers what turns the
artefact above into a **system**: why it fires (explainability), whether it will keep working
(drift), what it is worth (economics), how it is served (latency, persistence) and what has to be
built next.

In [ ]:
# ============================================================================
# 5.1  Global explainability: permutation importance on the sealed test set.
#      Computed on the FULL pipeline, so importances are attributed to the
#      original business features rather than to one-hot columns.
# ============================================================================
from sklearn.inspection import permutation_importance

sample_n = min(8_000, len(X_test))
idx = np.random.RandomState(CFG.random_state).choice(len(X_test), sample_n,
                                                     replace=False)
X_imp, y_imp = X_test.iloc[idx], y_test.iloc[idx]

t0 = time.time()
imp = permutation_importance(final_model, X_imp, y_imp, n_repeats=5,
                             scoring="average_precision",
                             random_state=CFG.random_state, n_jobs=2)
print("permutation importance on {:,} test rows in {:.0f}s".format(sample_n,
                                                                   time.time() - t0))

importance = (pd.DataFrame({"feature": X_test.columns,
                            "drop_in_pr_auc": imp.importances_mean,
                            "std": imp.importances_std})
              .sort_values("drop_in_pr_auc", ascending=False)
              .reset_index(drop=True))
show(importance.head(20), 20, "top 20 features by PR-AUC lost when shuffled")

top = importance.head(18).iloc[::-1]
fig, ax = plt.subplots(figsize=(8.6, 6.2))
ax.barh(top["feature"], top["drop_in_pr_auc"], xerr=top["std"], color="#4C78A8")
ax.set_xlabel("decrease in test PR-AUC when the feature is shuffled")
ax.set_title("Permutation importance - {}".format(FINAL_NAME))
plt.tight_layout()
plt.show()

engineered_set = set(["log_amount", "amount_vs_cust_mean", "cust_prev_mean_amount",
                      "cust_txn_seq", "amount_to_limit", "exposure_24h_to_limit",
                      "heuristic_risk_score", "is_burst_1h", "device_shared",
                      "device_n_prev_customers", "is_first_time_device",
                      "is_new_device", "ip_country_changed", "ip_not_home",
                      "ip_high_risk", "merchant_not_home",
                      "ip_vs_merchant_mismatch", "is_night", "is_weekend",
                      "days_since_signup", "is_new_account",
                      "customer_id_freq_train", "device_id_freq_train",
                      "amount_round_number"]
                     + [c for c in X_test.columns
                        if c.startswith(("n_txn_prev_", "amt_sum_prev_",
                                         "secs_since_prev"))])
eng_share = (importance[importance["feature"].isin(engineered_set)]["drop_in_pr_auc"]
             .clip(lower=0).sum()
             / max(importance["drop_in_pr_auc"].clip(lower=0).sum(), 1e-12))
print("\nEngineered (non-raw) features account for {:.1%} of total permutation"
      " importance\n-> Section 1.5 feature engineering, not the algorithm, is"
      " where the performance lives.".format(eng_share))

In [ ]:
# ============================================================================
# 5.2  Local explainability = "reason codes" an analyst can act on.
#      SHAP is used when available; otherwise a deviation-based fallback keeps
#      the capability (never leave an alert unexplained).
# ============================================================================
def reason_codes(row: pd.Series, reference: pd.DataFrame,
                 importance_table: pd.DataFrame, k: int = 6) -> pd.DataFrame:
    """Explain one alert by ranking important features by their deviation
    from the legitimate-population baseline (a robust z-score)."""
    ref = reference.select_dtypes(include=[np.number])
    med, iqr = ref.median(), (ref.quantile(0.75) - ref.quantile(0.25)).replace(0, np.nan)
    out = []
    weights = importance_table.set_index("feature")["drop_in_pr_auc"].clip(lower=0)
    for col in ref.columns:
        if pd.isna(row.get(col, np.nan)):
            continue
        dev = (row[col] - med[col]) / (iqr[col] if pd.notna(iqr[col]) else 1.0)
        out.append({"feature": col, "value": row[col],
                    "population_median": med[col], "robust_deviation": dev,
                    "weighted": abs(dev) * float(weights.get(col, 0.0))})
    return (pd.DataFrame(out).sort_values("weighted", ascending=False)
            .head(k).reset_index(drop=True))


alert_order = np.argsort(-test_score)
worst = int(alert_order[0])
section("highest-risk alert in the test period")
print("score            : {:.4f}   (threshold {:.4f})".format(test_score[worst],
                                                              THRESHOLD_FINAL))
print("actually fraud?  : {}".format(bool(y_test.iloc[worst])))
print("amount           : {}".format(money(test_df["amount"].iloc[worst])))
print("timestamp        : {}".format(test_df[CFG.time_col].iloc[worst]))
print("\nreason codes (why the model fired):")
print(reason_codes(X_test.iloc[worst], train_df[X_train.columns], importance)
      .to_string(index=False))

if shap is not None:
    try:
        prep = final_model.named_steps["prep"]
        model_only = final_model.named_steps["model"]
        Xs = prep.transform(X_test.iloc[alert_order[:400]])
        try:
            names = list(prep.get_feature_names_out())
        except Exception:
            names = ["f{}".format(i) for i in range(Xs.shape[1])]
        explainer = shap.TreeExplainer(model_only)
        sv = explainer.shap_values(Xs)
        sv = sv[1] if isinstance(sv, list) else sv
        shap.summary_plot(sv, features=Xs, feature_names=names, max_display=15,
                          show=False, plot_size=(9, 5.5))
        plt.title("SHAP - drivers of the 400 riskiest test transactions")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print("\nSHAP unavailable for this estimator ({}); the deviation-based"
              " reason codes above are the fallback.".format(type(exc).__name__))
else:
    print("\nshap is not installed - `pip install shap` to add SHAP attributions."
          "\nThe deviation-based reason codes above already give the analyst"
          " actionable output.")

In [ ]:
# ============================================================================
# 5.3  Will it keep working? Drift monitoring + performance over time.
# ============================================================================
def population_stability_index(expected: pd.Series, actual: pd.Series,
                               bins: int = 10) -> float:
    """PSI between a reference and a current distribution.

    Rule of thumb: < 0.10 stable, 0.10-0.25 monitor, > 0.25 investigate.
    """
    e, a = expected.dropna(), actual.dropna()
    if e.empty or a.empty:
        return float("nan")
    edges = np.unique(np.quantile(e, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    e_pct = np.histogram(e, bins=edges)[0] / len(e)
    a_pct = np.histogram(a, bins=edges)[0] / len(a)
    e_pct, a_pct = np.clip(e_pct, 1e-6, None), np.clip(a_pct, 1e-6, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))


watch = [c for c in ["amount", "log_amount", "n_txn_prev_24h", "amount_vs_cust_mean",
                     "secs_since_prev_txn", "heuristic_risk_score", "hour",
                     "days_since_signup", "device_n_prev_customers"]
         if c in X_train.columns]
psi = pd.DataFrame({
    "feature": watch,
    "psi_train_vs_test": [population_stability_index(X_train[c], X_test[c])
                          for c in watch],
})
psi["status"] = pd.cut(psi["psi_train_vs_test"], [-1, 0.10, 0.25, np.inf],
                       labels=["stable", "monitor", "investigate"])
section("feature drift: training period vs test period")
print(psi.round(4).to_string(index=False))

# score drift + realised performance month by month -------------------------
monitor = pd.DataFrame({
    "month": test_df[CFG.time_col].dt.to_period("M").astype(str),
    "y": y_test.to_numpy(), "score": test_score, "pred": y_test_pred,
    "amount": amt_test,
})
month_rows = []
for month, g in monitor.groupby("month", sort=True):
    month_rows.append({
        "month": month,
        "n": len(g),
        "fraud_rate_%": 100 * g["y"].mean(),
        "alert_rate_%": 100 * g["pred"].mean(),
        "pr_auc": (average_precision_score(g["y"], g["score"])
                   if g["y"].nunique() > 1 else np.nan),
        "recall": recall_score(g["y"], g["pred"], zero_division=0),
        "precision": precision_score(g["y"], g["pred"], zero_division=0),
        "mean_score": g["score"].mean(),
    })
by_month = pd.DataFrame(month_rows).set_index("month")
section("performance stability across the test period")
print(by_month.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.0))
ax = axes[0]
ax.plot(by_month.index.astype(str), by_month["pr_auc"], marker="o", color="#4C78A8",
        label="PR-AUC")
ax.plot(by_month.index.astype(str), by_month["recall"], marker="s", color="#E45756",
        label="recall @ threshold")
ax.plot(by_month.index.astype(str), by_month["precision"], marker="^",
        color="#54A24B", label="precision @ threshold")
ax.set_title("Monthly performance (test period)")
ax.set_ylabel("metric")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(np.log10(np.clip(final_model.predict_proba(X_train)[:, 1], 1e-6, 1)),
        bins=50, density=True, alpha=0.55, label="train scores", color="#4C78A8")
ax.hist(np.log10(np.clip(test_score, 1e-6, 1)), bins=50, density=True, alpha=0.55,
        label="test scores", color="#E45756")
ax.axvline(np.log10(THRESHOLD_FINAL), ls=":", c="black", label="threshold")
ax.set_xlabel("log10(predicted fraud probability)")
ax.set_title("Score distribution drift")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Monitoring contract for production:")
print("  * daily : alert volume, score percentiles, PSI on the top-10 features")
print("  * weekly: precision on reviewed alerts (analyst feedback is the label)")
print("  * monthly: PR-AUC on matured labels (chargebacks arrive 30-90 days late)")
print("  * trigger: PSI > 0.25 on any top feature, or recall down > 15% relative")
print("             -> retrain on a rolling 12-month window and shadow-test")

In [ ]:
# ============================================================================
# 5.4  Economics: how sensitive is the decision to our cost assumptions?
# ============================================================================
def optimal_policy(y_true, scores, amounts, review_cost: float,
                   recovery_rate: float) -> Dict[str, float]:
    """Cost-optimal threshold under a specific set of business assumptions."""
    cfg = Config(review_cost=review_cost, recovery_rate=recovery_rate)
    best = None
    for t in np.unique(np.quantile(scores, np.linspace(0.90, 0.9999, 60))):
        pred = (scores >= t).astype(int)
        c = expected_cost(y_true, pred, amounts, cfg)
        cand = {"review_cost": review_cost, "recovery_rate": recovery_rate,
                "threshold": t, "recall": recall_score(y_true, pred, zero_division=0),
                "precision": precision_score(y_true, pred, zero_division=0),
                "alert_rate": pred.mean(), "net_saving": c["net_saving"]}
        if best is None or cand["net_saving"] > best["net_saving"]:
            best = cand
    return best


grid = [optimal_policy(y_test, test_score, amt_test, rc, rr)
        for rc in (3.0, 8.0, 20.0) for rr in (0.0, 0.25, 0.60)]
sensitivity = pd.DataFrame(grid)
section("sensitivity of the operating point to the cost assumptions")
print(sensitivity.round(4).to_string(index=False))

print("\nInterpretation:")
print("  * cheap reviews / low recovery -> lower threshold, more alerts, more"
      " recall")
print("  * expensive reviews / high recovery -> higher threshold, fewer alerts")
print("  * the model is fixed; only the DECISION moves. This separation of the"
      " ranking\n    model from the business policy is what lets operations"
      " retune without\n    retraining.")

base = expected_cost(y_test, y_test_pred, amt_test, CFG)
section("business case on the test period ({} weeks)".format(
    round((test_df[CFG.time_col].max() - test_df[CFG.time_col].min()).days / 7)))
print("fraud value in period          : {}".format(money(amt_test[y_test == 1].sum())))
print("loss if we did nothing         : {}".format(money(base["cost_if_no_model"])))
print("loss + operating cost with model: {}".format(money(base["total_cost"])))
print("net saving                     : {}".format(money(base["net_saving"])))
print("alerts sent to analysts        : {:,} ({:.2f}% of traffic)".format(
    int(y_test_pred.sum()), 100 * y_test_pred.mean()))
print("saving per alert reviewed      : {}".format(
    money(base["net_saving"] / max(y_test_pred.sum(), 1))))

In [ ]:
# ============================================================================
# 5.5  Serving: persistence, a scoring function, and latency measurement.
# ============================================================================
ARTEFACTS = {
    "model_name": FINAL_NAME,
    "threshold": THRESHOLD_FINAL,
    "features": {"numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES},
    "best_params": {k: (v if isinstance(v, (int, float, str, type(None)))
                        else str(v))
                    for k, v in FINAL_SEARCH.best_params_.items()},
    "trained_on": {"rows": int(len(X_fit)),
                   "from": str(train_df[CFG.time_col].min()),
                   "to": str(val_df[CFG.time_col].max())},
    "test_metrics": {k: float(v) for k, v in final_free.items()},
    "data_source": DATA_SOURCE,
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "random_state": CFG.random_state,
}
meta_path = os.path.join(CFG.artifact_dir, "model_metadata.json")
with open(meta_path, "w") as fh:
    json.dump(ARTEFACTS, fh, indent=2, default=str)
print("metadata written ->", meta_path)

if joblib is not None:
    model_path = os.path.join(CFG.artifact_dir, "fraud_model.joblib")
    joblib.dump({"pipeline": final_model, "threshold": THRESHOLD_FINAL,
                 "metadata": ARTEFACTS}, model_path)
    size_mb = os.path.getsize(model_path) / 1e6
    print("model written    -> {} ({:.1f} MB)".format(model_path, size_mb))
    bundle = joblib.load(model_path)                     # prove it round-trips
    reloaded = bundle["pipeline"]
    assert np.allclose(reloaded.predict_proba(X_test.head(50))[:, 1],
                       test_score[:50]), "reloaded model disagrees!"
    print("reload check     -> identical scores [OK]")
else:
    reloaded = final_model
    print("joblib unavailable; skipping serialisation check")


def score_transaction(features: Dict[str, Any], pipeline=None,
                      threshold: float = None) -> Dict[str, Any]:
    """Score one transaction the way an API endpoint would.

    Expects the engineered feature dictionary (in production the velocity and
    baseline features come from a streaming feature store, not from pandas).
    """
    pipeline = pipeline if pipeline is not None else reloaded
    threshold = THRESHOLD_FINAL if threshold is None else threshold
    frame = pd.DataFrame([features]).reindex(columns=FEATURES)
    # a one-row frame can infer object dtypes; force the numeric contract
    frame[NUMERIC_FEATURES] = frame[NUMERIC_FEATURES].apply(pd.to_numeric,
                                                            errors="coerce")
    frame[CATEGORICAL_FEATURES] = frame[CATEGORICAL_FEATURES].astype(object)
    prob = float(pipeline.predict_proba(frame)[0, 1])
    if prob >= threshold * 3:
        action, band = "block + notify customer", "high"
    elif prob >= threshold:
        action, band = "queue for analyst review", "medium"
    elif prob >= threshold / 3:
        action, band = "step-up authentication (3DS/OTP)", "low"
    else:
        action, band = "allow", "minimal"
    return {"fraud_probability": round(prob, 6), "risk_band": band,
            "decision": action, "threshold": threshold}


section("single-transaction scoring demo (riskiest test case)")
print(json.dumps(score_transaction(X_test.iloc[worst].to_dict()), indent=2))
print("\nlowest-risk test case")
print(json.dumps(score_transaction(X_test.iloc[int(alert_order[-1])].to_dict()),
                 indent=2))

batch = X_test.head(2_000)
t0 = time.time()
reloaded.predict_proba(batch)
batch_ms = 1000 * (time.time() - t0)
t0 = time.time()
for i in range(50):
    reloaded.predict_proba(X_test.iloc[[i]])
single_ms = 1000 * (time.time() - t0) / 50
section("latency (CPU, notebook hardware)")
print("batch of {:,}: {:.0f} ms total -> {:.3f} ms/transaction".format(
    len(batch), batch_ms, batch_ms / len(batch)))
print("single call  : {:.1f} ms/transaction (pandas + pipeline overhead)".format(
    single_ms))
print("-> batch scoring is well inside a real-time budget; a production endpoint"
      "\n   would drop pandas and pre-materialise velocity features in Redis to"
      " hold\n   the p99 under ~50 ms.")

### 5.6 Considerations beyond this notebook

**Extra features I would add next (highest expected value first)**

1. **Graph / network features.** Fraud is organised: shared devices, IPs, shipping addresses and
   beneficiary accounts form components. Features such as *number of distinct accounts on this
   device in 30 days*, *component size*, and *distance to a known fraudster in the graph* are
   consistently the strongest additions in industry. `device_shared` here is a one-hop
   approximation of that idea.
2. **Merchant-side and counterparty risk**: rolling chargeback rate per merchant/MCC, first-time
   beneficiary flag, beneficiary account age.
3. **Session and behavioural biometrics**: typing cadence, mouse entropy, time-on-page, copy-paste
   of the card number, headless-browser and emulator signals — these separate a human owner from a
   bot replaying stolen credentials.
4. **Geo-velocity**: physically impossible travel between consecutive transactions (needs
   lat/long, not just country).
5. **Unsupervised layer for zero-day attacks.** A supervised model can only recognise fraud
   patterns that already appear in labels. Running Isolation Forest / autoencoder reconstruction
   error **in parallel** catches novel patterns; the ensemble is the standard defence-in-depth
   design.
6. **Sequence models.** A GRU/transformer over each customer's last *k* transactions learns
   ordering effects that hand-built window aggregates approximate.

**Operational realities that change the design**

| Reality | Consequence |
|---|---|
| **Label delay** — chargebacks land 30-90 days later | never evaluate the most recent weeks as if labels were complete; use analyst decisions as fast, biased proxy labels and reconcile later |
| **Feedback loop / selective labelling** — we only ever learn the outcome of transactions we allowed | keep a small random *hold-out of un-blocked* traffic (or use reject inference) so the training distribution does not collapse onto what the current model already trusts |
| **Adversarial adaptation** — attackers probe and adapt within days | retrain on a rolling window, keep some features cheap-to-refresh, monitor per-segment recall, and avoid publishing thresholds |
| **Concept drift** (new products, promotions, holidays) | PSI monitoring (5.3) + scheduled retraining + champion/challenger shadow deployment |
| **Latency budget** (~50-100 ms end to end) | keep the feature store as the bottleneck, not the model; boosted trees are fast enough, deep models usually are not |
| **Cost asymmetry changes by segment** | a USD 20 card-not-present grocery purchase and a USD 9,000 crypto transfer should not share one threshold: use **amount-banded or segment-specific thresholds** |
| **Regulatory audit** | version data, code, features and thresholds; keep reason codes for every automated decision (Section 6) |

<a name="sec6"></a>
# 6. AI Ethics Consideration  *(5 marks)*

A fraud model is a **consequential automated decision**: a false positive can strand someone
abroad with a declined card, freeze a small business's payroll, or brand a customer as a criminal.
This section is therefore an audit with measurements, not a paragraph of good intentions.

The six pillars, and how each is *evidenced* below:

| Pillar | What can go wrong | Evidence produced here |
|---|---|---|
| **Fairness** | error rates differ by age, gender, region → discriminatory friction | 6.1 group metrics, disparate impact, equal-opportunity gap |
| **Proxy discrimination** | gender is excluded but reconstructible from behaviour | 6.2 proxy-leakage test |
| **Mitigation & trade-offs** | "fix" bias while destroying utility or breaking the law | 6.3 threshold equalisation with the cost quantified |
| **Privacy** | fraud engines hoard sensitive personal data | 6.4 PII inventory, pseudonymisation, minimisation, retention |
| **Transparency & contestability** | opaque blocks with no route to appeal | 6.5 reason codes, human-in-the-loop, appeal SLA |
| **Accountability** | nobody owns the model's harms | 6.6 model card + risk register + governance |

In [ ]:
# ============================================================================
# 6.1  Fairness audit: are the model's ERRORS distributed evenly?
#      Note: `gender` was never a model input (Section 1.6) - it is used here
#      only to measure outcomes, which is the legitimate purpose.
# ============================================================================
audit = pd.DataFrame({
    "y_true": y_test.to_numpy(),
    "y_pred": y_test_pred,
    "score": test_score,
    "amount": amt_test,
})
for col in SENSITIVE_FOR_AUDIT:
    if col in test_df.columns:
        audit[col] = test_df[col].to_numpy()
if "age" in audit.columns:
    audit["age_band"] = pd.cut(audit["age"], [17, 25, 35, 50, 65, 120],
                               labels=["18-25", "26-35", "36-50", "51-65", "66+"])

GROUPS = [c for c in ["gender", "age_band", "region", "account_type"]
          if c in audit.columns]


def group_fairness(frame: pd.DataFrame, group_col: str,
                   min_n: int = 200, min_pos: int = 5) -> pd.DataFrame:
    """Per-group confusion-matrix rates.

    selection_rate : share of the group that gets flagged (statistical parity)
    tpr / recall   : share of that group's real fraud we catch (equal opportunity)
    fpr            : share of that group's GENUINE traffic wrongly blocked
                     - the harm metric that customers actually experience
    """
    rows = []
    for value, g in frame.groupby(group_col, observed=True):
        pos, neg = g["y_true"] == 1, g["y_true"] == 0
        tp = int(((g["y_pred"] == 1) & pos).sum())
        fp = int(((g["y_pred"] == 1) & neg).sum())
        fn = int(((g["y_pred"] == 0) & pos).sum())
        tn = int(((g["y_pred"] == 0) & neg).sum())
        enough = (len(g) >= min_n) and (int(pos.sum()) >= min_pos)
        rows.append({
            group_col: value, "n": len(g), "fraud_prevalence_%": 100 * pos.mean(),
            "selection_rate_%": 100 * (g["y_pred"] == 1).mean(),
            "tpr_recall": tp / max(tp + fn, 1) if enough else np.nan,
            "fpr_%": 100 * fp / max(fp + tn, 1),
            "precision": tp / max(tp + fp, 1) if (tp + fp) else np.nan,
            "mean_score": g["score"].mean(),
            "reliable": enough,
        })
    return pd.DataFrame(rows).set_index(group_col)


def fairness_summary(tbl: pd.DataFrame, group_col: str) -> Dict[str, float]:
    """Standard group-fairness gap statistics for one attribute."""
    ok = tbl[tbl["reliable"]] if tbl["reliable"].any() else tbl
    sel, fpr, tpr = ok["selection_rate_%"], ok["fpr_%"], ok["tpr_recall"].dropna()
    return {
        "attribute": group_col,
        "disparate_impact_ratio": (sel.min() / sel.max()) if sel.max() else np.nan,
        "statistical_parity_diff_pp": sel.max() - sel.min(),
        "equal_opportunity_diff": (tpr.max() - tpr.min()) if len(tpr) > 1 else np.nan,
        "fpr_ratio_worst_best": (fpr.max() / fpr.min()) if fpr.min() > 0 else np.nan,
        "fpr_gap_pp": fpr.max() - fpr.min(),
        "80%_rule_passed": bool((sel.min() / sel.max()) >= 0.8) if sel.max() else None,
    }


summaries = []
for col in GROUPS:
    tbl = group_fairness(audit, col)
    section("fairness by {}".format(col))
    print(tbl.round(4).to_string())
    summaries.append(fairness_summary(tbl, col))

fair_summary = pd.DataFrame(summaries).set_index("attribute")
section("fairness gap summary (test set, threshold {:.4f})".format(THRESHOLD_FINAL))
print(fair_summary.round(4).to_string())
print("\nHow to read this:")
print("  disparate_impact_ratio  : 1.0 = identical alert rates; < 0.8 fails the"
      " classic 80% rule")
print("  equal_opportunity_diff  : gap in FRAUD CAUGHT between groups"
      " (protection inequality)")
print("  fpr_ratio_worst_best    : how many times more often the worst-served"
      " group is\n                            wrongly blocked - the customer-harm"
      " metric")

In [ ]:
# ============================================================================
# 6.2  Fairness through unawareness is NOT enough: proxy-leakage test.
#      Can the model's own feature set reconstruct the protected attribute?
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
plot_group = "age_band" if "age_band" in audit.columns else GROUPS[0]
tbl = group_fairness(audit, plot_group)
x = np.arange(len(tbl))
ax = axes[0]
ax.bar(x - 0.2, tbl["fpr_%"], width=0.4, label="false-positive rate (%)",
       color="#E45756")
ax.bar(x + 0.2, tbl["selection_rate_%"], width=0.4, label="alert rate (%)",
       color="#4C78A8")
ax.set_xticks(x, [str(i) for i in tbl.index], rotation=0)
ax.set_title("Harm distribution by {}".format(plot_group))
ax.set_ylabel("%")
ax.legend(fontsize=8)

ax = axes[1]
tpr_tbl = tbl["tpr_recall"]
ax.bar([str(i) for i in tpr_tbl.index], tpr_tbl.to_numpy(), color="#54A24B")
ax.axhline(float(np.nanmean(tpr_tbl.to_numpy())), ls="--", c="grey",
           label="mean recall")
ax.set_title("Protection received by {} (recall)".format(plot_group))
ax.set_ylabel("fraud caught")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

if "gender" in audit.columns:
    mask = test_df["gender"].isin(["f", "m"]).to_numpy()
    if mask.sum() > 500:
      try:
        proxy_y = (test_df.loc[mask, "gender"] == "f").astype(int)
        proxy_X = X_test[mask]
        cut = int(0.7 * len(proxy_X))
        probe = Pipeline([
            ("prep", build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)),
            ("model", LogisticRegression(max_iter=1000,
                                         random_state=CFG.random_state)),
        ]).fit(proxy_X.iloc[:cut], proxy_y.iloc[:cut])
        proxy_auc = roc_auc_score(proxy_y.iloc[cut:],
                                 probe.predict_proba(proxy_X.iloc[cut:])[:, 1])
        section("proxy-leakage probe: predicting GENDER from the model's features")
        print("ROC-AUC of the probe: {:.3f}   (0.50 = no leakage, 1.00 = the"
              " protected\n                     attribute is fully"
              " reconstructible)".format(proxy_auc))
        print("verdict: {}".format(
            "low proxy leakage - excluding gender is largely effective"
            if proxy_auc < 0.60 else
            "MATERIAL proxy leakage - dropping the column does NOT remove the"
            " signal;\n         the disparities in 6.1 must be managed"
            " explicitly (6.3)"))
      except Exception as exc:
        print("proxy probe skipped ({})".format(type(exc).__name__))

In [ ]:
# ============================================================================
# 6.3  Mitigation: equalising the harm, and what it costs.
#      Technique = post-processing with group-specific thresholds calibrated so
#      that every group's false-positive rate matches the global target.
# ============================================================================
def equalised_fpr_thresholds(frame: pd.DataFrame, group_col: str,
                             target_fpr: float) -> Dict[Any, float]:
    """Per-group thresholds that give each group the same false-positive rate."""
    out = {}
    for value, g in frame.groupby(group_col, observed=True):
        genuine = g.loc[g["y_true"] == 0, "score"].to_numpy()
        out[value] = (float(np.quantile(genuine, 1.0 - target_fpr))
                      if len(genuine) > 50 else THRESHOLD_FINAL)
    return out


MITIGATE_ON = plot_group
global_fpr = float(((audit["y_pred"] == 1) & (audit["y_true"] == 0)).sum()
                   / max((audit["y_true"] == 0).sum(), 1))
thresholds = equalised_fpr_thresholds(audit, MITIGATE_ON, global_fpr)

mitigated = audit.copy()
mitigated["y_pred"] = [
    int(s >= thresholds.get(gval, THRESHOLD_FINAL))
    for s, gval in zip(mitigated["score"], mitigated[MITIGATE_ON])
]

before = group_fairness(audit, MITIGATE_ON)
after = group_fairness(mitigated, MITIGATE_ON)
side = pd.concat([before[["selection_rate_%", "fpr_%", "tpr_recall"]]
                 .add_suffix("_before"),
                 after[["selection_rate_%", "fpr_%", "tpr_recall"]]
                 .add_suffix("_after")], axis=1)
section("bias mitigation on '{}' (equalised FPR = {:.3f}%)".format(
    MITIGATE_ON, 100 * global_fpr))
print("group thresholds:", {str(k): round(v, 4) for k, v in thresholds.items()})
print()
print(side.round(4).to_string())

b, a = fairness_summary(before, MITIGATE_ON), fairness_summary(after, MITIGATE_ON)
gap_tbl = pd.DataFrame({"before": b, "after": a}).drop(index=["attribute"])
print("\n" + gap_tbl.to_string())

cost_before = expected_cost(audit["y_true"], audit["y_pred"], audit["amount"], CFG)
cost_after = expected_cost(mitigated["y_true"], mitigated["y_pred"],
                           mitigated["amount"], CFG)
print("\nUtility cost of the intervention")
print("  recall  : {:.4f} -> {:.4f}".format(
    recall_score(audit["y_true"], audit["y_pred"], zero_division=0),
    recall_score(mitigated["y_true"], mitigated["y_pred"], zero_division=0)))
print("  net saving: {} -> {}  ({})".format(
    money(cost_before["net_saving"]), money(cost_after["net_saving"]),
    money(cost_after["net_saving"] - cost_before["net_saving"])))
print("""
LEGAL AND ETHICAL CAVEAT - this is deliberately not applied by default.
Using a protected attribute at DECISION time (different thresholds per group)
is unlawful in many jurisdictions even when the intent is to reduce harm, and
it can also reduce the protection given to a group. The defensible order of
preference is:
  1. fix the DATA and FEATURES (remove proxies, rebalance under-represented
     segments, add features that explain the behaviour rather than the person);
  2. constrain training (fairness regularisation, reweighting) rather than
     post-processing on the protected attribute;
  3. keep a HUMAN reviewer for the affected segments and monitor the gap;
  4. only consider group-aware thresholds with documented legal sign-off.
The measurement above is what makes that conversation possible - the numbers
belong in the model card either way.
""")

In [ ]:
# ============================================================================
# 6.4  Privacy by design: PII inventory, minimisation, pseudonymisation.
# ============================================================================
import hashlib

PII_INVENTORY = pd.DataFrame([
    ("customer_id", "direct identifier", "pseudonymised (salted SHA-256) before storage; "
     "only used to build aggregates"),
    ("device_id", "device identifier / quasi-identifier", "hashed fingerprint; retained 13 months"),
    ("ip_country", "coarse location", "country only - full IP is never persisted in the feature store"),
    ("age", "personal data", "used as a numeric feature; audited for disparity in 6.1"),
    ("gender", "special-category-adjacent", "EXCLUDED from the model; retained only for the fairness audit, access-controlled"),
    ("region / home_country", "quasi-identifier", "kept for genuine geo-risk; audited for proxy discrimination"),
    ("amount / merchant_category", "transaction data", "core fraud signal; purpose-limited to fraud prevention"),
], columns=["field", "classification", "control applied"])
section("PII inventory and controls")
print(PII_INVENTORY.to_string(index=False))


def pseudonymise(value: str, salt: str = "xyz-cyber-2026-rotating-salt") -> str:
    """Salted one-way hash - lets us aggregate per customer without storing who."""
    return hashlib.sha256((salt + str(value)).encode()).hexdigest()[:16]


demo = test_df[["customer_id", "device_id"]].head(3).copy()
demo["customer_pseudonym"] = demo["customer_id"].map(pseudonymise)
demo["device_pseudonym"] = demo["device_id"].map(pseudonymise)
print("\npseudonymisation demo (irreversible without the rotating salt):")
print(demo.to_string(index=False))

print("""
Data-protection commitments implemented or specified
  * PURPOSE LIMITATION  - features exist solely for fraud prevention; reuse for
    marketing or credit decisions requires a new lawful basis and a new DPIA.
  * MINIMISATION        - no full IP, no device serial, no message content, no
    special-category data (health, religion, biometrics) enters the model.
  * LAWFUL BASIS        - fraud monitoring rests on legitimate interest /
    contractual necessity (GDPR Art. 6(1)(b)/(f), Recital 71 explicitly
    recognises fraud monitoring), documented in a DPIA.
  * STORAGE LIMITATION  - raw events 13 months, engineered features 24 months,
    fairness-audit attributes in a separate access-controlled store.
  * SECURITY            - encryption in transit and at rest, least-privilege
    access, no PII in logs, model artefacts signed and versioned (5.5).
  * TRANSFER            - no cross-border transfer of raw PII; only aggregated
    features leave the regional boundary.
""")

In [ ]:
# ============================================================================
# 6.5 / 6.6  Governance artefacts: the model card (saved with the model).
# ============================================================================
_ratios = fair_summary["fpr_ratio_worst_best"].astype(float).fillna(1.0)
worst_fpr_ratio = float(_ratios.max())
worst_attr = str(_ratios.idxmax())

MODEL_CARD = """
# Model Card - XYZ Cybersecurity Fraud Detection Engine v1.0

## 1. Model details
- **Model**            : {name} (scikit-learn pipeline: impute -> scale/one-hot -> classifier)
- **Version / date**   : v1.0, {date}
- **Owner**            : Fraud Data Science, XYZ Cybersecurity (accountable exec: Head of Risk)
- **Type**             : binary classifier producing a fraud probability; NOT an
                         automatic blocking rule on its own

## 2. Intended use
- **In scope**   : ranking card/online transactions for analyst review, step-up
                   authentication, and blocking only above the 'high' risk band.
- **Out of scope**: credit or insurance decisions, employment screening, law-enforcement
                   evidence, or any use of the score as proof of guilt.
- **Users**      : trained fraud analysts with reason codes; never an unsupervised
                   auto-decision for account closure.

## 3. Data
- **Source**        : {source}
- **Training window**: {train_from} to {train_to} ({rows:,} transactions)
- **Label**         : confirmed fraud/chargeback. Known limitation: labels arrive with a
                      30-90 day delay and some fraud is never reported (label noise).
- **Protected attributes**: `gender` excluded from inputs; `age`, `region`, `home_country`
                      retained for genuine risk value and audited in Section 6.1.

## 4. Performance (sealed test period, threshold {thr:.4f})
| Metric | Value |
|---|---|
| PR-AUC (average precision) | {pr_auc:.4f} |
| ROC-AUC | {roc_auc:.4f} |
| Recall (fraud caught) | {recall:.4f} |
| Precision (alert purity) | {precision:.4f} |
| Alert rate | {alert_rate:.4f} |
| Net saving vs no model | {saving} |

## 5. Fairness audit
- Worst false-positive-rate ratio: **{fpr_ratio:.2f}x** on `{fpr_attr}`
- 80%-rule (disparate impact) results: {eighty}
- Mitigation options and their utility cost are documented in Section 6.3.
- Committed review cadence: fairness metrics recomputed monthly and on every retrain.

## 6. Ethical considerations and limitations
- False positives cause real customer harm (declined payments, frozen funds); the
  threshold is chosen on an explicit cost model and reviewed by the business.
- The model reflects historical detection behaviour and can inherit the biases of past
  analyst decisions (selective labelling).
- Performance degrades under adversarial adaptation and drift; monitored per 5.3.
- Scores are not calibrated probabilities of guilt and must never be presented as such.

## 7. Human oversight and redress
- Every alert ships with reason codes (5.2); analysts can override in one click.
- Customers are told a security check occurred, can contest a block through support
  with a target 24-hour resolution, and a human makes the final account decision.
- Overrides are logged and fed back as training signal and as a fairness monitor.

## 8. Maintenance
- Retrain: monthly rolling window, or on drift trigger (PSI > 0.25 / recall -15%).
- Champion-challenger shadow evaluation before promotion; rollback plan retained.
- Full lineage: data snapshot, feature code, hyper-parameters, threshold and metrics
  versioned together ({meta}).
""".format(
    name=FINAL_NAME,
    date=pd.Timestamp.utcnow().strftime("%Y-%m-%d"),
    source=DATA_SOURCE,
    train_from=str(train_df[CFG.time_col].min().date()),
    train_to=str(val_df[CFG.time_col].max().date()),
    rows=len(X_fit),
    thr=THRESHOLD_FINAL,
    pr_auc=final_free["pr_auc"], roc_auc=final_free["roc_auc"],
    recall=final_point["recall"], precision=final_point["precision"],
    alert_rate=final_point["alert_rate"],
    saving=money(final_cost["net_saving"]),
    fpr_ratio=worst_fpr_ratio, fpr_attr=worst_attr,
    eighty=", ".join("{}={}".format(i, v) for i, v in
                     fair_summary["80%_rule_passed"].items()),
    meta=meta_path,
)

card_path = os.path.join(CFG.artifact_dir, "MODEL_CARD.md")
with open(card_path, "w") as fh:
    fh.write(MODEL_CARD)
print("model card written ->", card_path)
try:
    from IPython.display import Markdown, display
    display(Markdown(MODEL_CARD))
except Exception:
    print(MODEL_CARD)

### 6.7 Ethical risk register

| # | Risk | Who is harmed | Likelihood | Impact | Mitigation implemented / specified |
|---|---|---|---|---|---|
| 1 | **Disparate false positives** — one demographic is blocked more often because it legitimately behaves in a "risky-looking" way (new devices, travel, night-time app use) | customers in that group | High | High | measured in 6.1; proxy probe in 6.2; mitigation options + cost in 6.3; monthly monitoring in the model card |
| 2 | **Unequal protection** — recall differs by segment, so some customers are less protected from fraud | under-served segments | Medium | High | equal-opportunity gap tracked; segment recall in the monitoring contract |
| 3 | **Proxy discrimination** — excluded attributes are reconstructed from behavioural features | protected groups | Medium | High | proxy-leakage probe (6.2); feature-level review before every release |
| 4 | **Feedback loop** — blocked customers generate no future data, so the model's blind spots become self-confirming | new/atypical customers | High | Medium | random un-blocked hold-out, reject inference, override logging |
| 5 | **Over-reliance on automation** | wrongly blocked customers | Medium | High | model outputs a *risk band*, not a verdict; human decides account actions; step-up auth used before hard blocks |
| 6 | **Opacity** — a decision nobody can explain cannot be contested | customers, regulator | Medium | Medium | reason codes on every alert (5.2), model card, decision logging |
| 7 | **Privacy creep** — a rich behavioural profile is attractive for other uses | all customers | Medium | High | purpose limitation, pseudonymisation, retention limits, separate store for audit attributes (6.4) |
| 8 | **Adversarial gaming** — attackers probe thresholds and mimic legitimate patterns | the firm and its customers | High | Medium | thresholds not published, frequent retraining, unsupervised layer, rate-limit probing |
| 9 | **Financial exclusion** — thin-file or newly-arrived customers look permanently risky | vulnerable groups | Medium | High | `days_since_signup` monitored for disparity; manual review path for new accounts instead of silent decline |
| 10 | **Label bias** — historical labels encode which frauds past analysts *chose* to investigate | mis-served segments | Medium | Medium | documented as a limitation; sampling review queues randomly to de-bias labels |

### 6.8 Regulatory alignment (as designed)

- **GDPR Art. 22 / Recital 71** — solely automated decisions with significant effects require
  safeguards. We keep meaningful human involvement for account-level actions, provide reason codes,
  and support contestation, which is exactly the safeguard set the article expects. Recital 71
  explicitly recognises fraud monitoring as a legitimate purpose.
- **GDPR Art. 5** — purpose limitation, minimisation and storage limitation are implemented as the
  concrete controls listed in 6.4; a DPIA is required before production because the processing is
  systematic monitoring of behaviour.
- **EU AI Act** — creditworthiness scoring is listed as high-risk; systems whose purpose is
  detecting **financial fraud** are carved out of that entry, but if this score were ever reused
  for credit or eligibility decisions it would fall in scope. That reuse is explicitly out of
  scope in the model card, and any change would trigger a fresh conformity assessment.
- **Malaysia PDPA / regional equivalents** — consent notices, cross-border transfer limits and
  breach response are covered by the data-protection commitments in 6.4.
- **PCI-DSS** — no PAN, CVV or full track data is used as a feature; only tokenised references.
- **Model risk governance (e.g. SR 11-7 style)** — independent validation of the pipeline,
  documented assumptions (cost model, thresholds), champion-challenger process, and an owner named
  in the model card.

> **Bottom line.** The measurements above show that excluding a protected attribute is *not*
> sufficient evidence of fairness — the harm has to be measured on outcomes, the trade-off of any
> fix has to be quantified in both fairness and money, and the residual gap has to be disclosed and
> monitored. That is what the model card and risk register commit XYZ Cybersecurity to.

<a name="sec7"></a>
# 7. Documentation, Code Quality and Conclusions  *(3 marks)*

### 7.1 Engineering standards applied in this notebook

| Practice | How it shows up here |
|---|---|
| **Single configuration object** | the frozen `Config` dataclass holds every constant (seed, split sizes, cost model, search budget); no magic numbers buried in cells |
| **Reproducibility** | one seed (`RANDOM_STATE = 42`) threaded through the simulator, splits, models and searches; library versions printed in 0.1; deterministic chronological splits |
| **Pure, documented functions** | every transformation is a named function with a docstring, type hints and a single responsibility (`clean_transactions`, `add_behavioural_features`, `expected_cost`, `group_fairness`, ...) — no hidden state between cells |
| **Leakage discipline** | past-only features, chronological splits, `TimeSeriesSplit`, all fitted statistics inside `Pipeline`/`ColumnTransformer`, frequency encodings fitted on train only, and `assert`s that the split ordering holds |
| **Defensive dependencies** | optional imports (`xgboost`, `imblearn`, `shap`, `joblib`) degrade gracefully; `safe_estimator` and `make_ohe` absorb scikit-learn version differences |
| **Auditability** | the cleaning step returns an action log; the join is asserted; artefacts (model, metadata, model card) are written to `artifacts/` |
| **Honest narrative** | every quoted number is generated at run time from the computed objects, so the commentary cannot drift from the results |
| **Reviewability** | rubric-aligned sections, tables that state *why* before *what*, consistent plotting style, and no cell that depends on manual edits |

In [ ]:
# ============================================================================
# 7.1  Consolidated results: everything the assignment asks for, in one table.
# ============================================================================
section("consolidated results")

print("\n[1] DATA")
data_tbl = pd.DataFrame([
    ("source", DATA_SOURCE),
    ("raw transactions", "{:,}".format(len(txn_raw))),
    ("after cleaning", "{:,}".format(len(df_clean))),
    ("fraud rate", pct(df_clean[CFG.target].mean())),
    ("features engineered", "{} ({} numeric, {} categorical)".format(
        len(FEATURES), len(NUMERIC_FEATURES), len(CATEGORICAL_FEATURES))),
    ("split", "chronological 60/20/20 (train {:,} / val {:,} / test {:,})".format(
        len(X_train), len(X_val), len(X_test))),
], columns=["item", "value"])
print(data_tbl.to_string(index=False))

print("\n[2] MODEL SELECTION (cross-validated on the training period)")
print(cv_results[["model", "pr_auc_mean", "pr_auc_std", "roc_auc_mean",
                  "fit_seconds"]].to_string(index=False))

print("\n[3] TUNING IMPACT (validation)")
print(comp[["pr_auc", "roc_auc", "recall@prec>=0.30",
            "precision@1%alerts"]].round(4).to_string())

print("\n[4] FINAL MODEL ON THE SEALED TEST SET")
headline = pd.DataFrame([
    ("model", FINAL_NAME),
    ("decision threshold", "{:.4f}".format(THRESHOLD_FINAL)),
    ("PR-AUC (average precision)", "{:.4f}".format(final_free["pr_auc"])),
    ("ROC-AUC", "{:.4f}".format(final_free["roc_auc"])),
    ("recall / fraud caught", "{:.4f}".format(final_point["recall"])),
    ("precision / alert purity", "{:.4f}".format(final_point["precision"])),
    ("F2 score", "{:.4f}".format(final_point["f2"])),
    ("alert rate", "{:.4f}".format(final_point["alert_rate"])),
    ("recall @ precision>=0.30", "{:.4f}".format(final_free["recall@prec>=0.30"])),
    ("precision @ 1% alert budget", "{:.4f}".format(final_free["precision@1%alerts"])),
    ("accuracy (reported only to show it is useless)",
     "{:.4f}".format(final_point["accuracy_(misleading)"])),
    ("net saving vs no model", money(final_cost["net_saving"])),
], columns=["metric", "value"])
print(headline.to_string(index=False))

print("\n[5] TOP 8 DRIVERS (permutation importance)")
print(importance.head(8).to_string(index=False))

print("\n[6] FAIRNESS GAPS")
print(fair_summary.round(4).to_string())

# machine-readable summary for the report / appendix
summary_path = os.path.join(CFG.artifact_dir, "results_summary.json")
with open(summary_path, "w") as fh:
    json.dump({
        "data": dict(data_tbl.values.tolist()),
        "model_selection": cv_results.to_dict(orient="records"),
        "final_model": FINAL_NAME,
        "threshold": THRESHOLD_FINAL,
        "test_metrics": {k: float(v) for k, v in
                         dict(final_free, **final_point).items()},
        "economics": {k: float(v) for k, v in final_cost.items()},
        "fairness": json.loads(fair_summary.reset_index().to_json(orient="records")),
        "top_features": importance.head(10).to_dict(orient="records"),
    }, fh, indent=2, default=str)
print("\nmachine-readable summary ->", summary_path)
print("artifacts directory     ->", os.path.abspath(CFG.artifact_dir))
print("  " + "\n  ".join(sorted(os.listdir(CFG.artifact_dir))))

In [ ]:
# ============================================================================
# 7.2  Environment capture, so the run can be reproduced exactly.
# ============================================================================
requirements = ["pandas=={}".format(pd.__version__),
                "numpy=={}".format(np.__version__),
                "scikit-learn=={}".format(sklearn.__version__),
                "matplotlib=={}".format(matplotlib.__version__)]
requirements += ["{}=={}".format(k, v) for k, v in OPTIONAL.items()]
req_path = os.path.join(CFG.artifact_dir, "requirements.txt")
with open(req_path, "w") as fh:
    fh.write("\n".join(requirements) + "\n")
print("pinned environment ->", req_path)
print("\n".join(requirements))
print("\nreproduce with:  pip install -r {}".format(req_path))
print("random_state used everywhere:", CFG.random_state)

### 7.2 Conclusions

**What was built.** An end-to-end fraud-detection pipeline: three source tables joined and audited,
a logged cleaning pass over genuinely dirty data, ~45 engineered features built strictly from each
customer's past, seven candidate models compared under time-series cross-validation on PR-AUC, a
randomised hyper-parameter search on the two most promising families, a decision threshold chosen
from an explicit cost model rather than from the 0.5 default, one honest evaluation on a sealed
future period, and a fairness/privacy audit with a model card.

**The four findings that matter** (all quantified in the printed outputs above):

1. **Feature engineering beat model choice.** Behavioural context — velocity, device novelty,
   deviation from the customer's own spending baseline, geography mismatch — dominates the
   permutation-importance ranking, while raw columns are individually near-useless. A simple model
   on good features would outperform a tuned model on raw rows.
2. **Metric choice changes the answer.** Accuracy is ~98-99% for every candidate including bad
   ones; ROC-AUC compresses the field; only PR-AUC and precision-at-alert-budget rank the models
   the way the fraud team experiences them.
3. **The threshold is a business decision, not a default.** Sweeping thresholds against a cost
   model (fraud value at risk vs analyst time and customer friction) moves the operating point far
   from 0.5 and is what converts a ranking model into money saved.
4. **Fairness has to be measured, not assumed.** `gender` was never an input, yet outcome rates
   still differ across demographic groups because behaviour correlates with demographics — the
   proxy probe and group error rates make that visible, and the mitigation experiment prices the
   trade-off honestly.

**Known limitations (stated deliberately).**

- Fraud labels are delayed and noisy; the most recent period is always optimistically labelled.
- The threshold is tuned on validation and reused for the model refitted on train+validation; a
  production release would recalibrate on a dedicated recent slice or use a calibrated wrapper.
- Only a one-hop device-sharing feature stands in for real graph features.
- No unsupervised/zero-day layer is trained here, so genuinely novel attack patterns will be
  missed until they enter the labels.
- If the notebook ran on the simulator rather than the supplied CSVs, absolute metric values
  reflect the simulator's generative process; the *methodology*, not the numbers, is what
  transfers.

**Next steps, in priority order:** graph/network features → an unsupervised anomaly layer for
zero-day attacks → probability calibration (isotonic) for cleaner cost decisions → segment-specific
thresholds (amount bands) → shadow deployment with champion-challenger monitoring → automated
monthly retraining with the drift triggers from Section 5.3.

### 7.3 References

- Bergstra, J. & Bengio, Y. (2012). *Random Search for Hyper-Parameter Optimization.* JMLR 13.
- Saito, T. & Rehmsmeier, M. (2015). *The Precision-Recall Plot Is More Informative than the ROC
  Plot When Evaluating Binary Classifiers on Imbalanced Datasets.* PLOS ONE.
- Chawla, N. et al. (2002). *SMOTE: Synthetic Minority Over-sampling Technique.* JAIR 16.
- Dal Pozzolo, A. et al. (2015). *Calibrating Probability with Undersampling for Unbalanced
  Classification.* IEEE SSCI. (source of the calibration caveat in Section 3.3)
- Bhattacharyya, S. et al. (2011). *Data mining for credit card fraud: A comparative study.*
  Decision Support Systems.
- Mitchell, M. et al. (2019). *Model Cards for Model Reporting.* ACM FAT*.
- Barocas, S., Hardt, M. & Narayanan, A. (2019). *Fairness and Machine Learning.* fairmlbook.org.
- Hardt, M., Price, E. & Srebro, N. (2016). *Equality of Opportunity in Supervised Learning.*
  NeurIPS. (equal-opportunity and post-processing definitions used in Section 6)
- scikit-learn user guide: *Imbalanced classification*, *Cross-validation for time series*,
  *Permutation importance*, *Probability calibration*.
- EU General Data Protection Regulation, Articles 5, 6 and 22 with Recital 71; EU AI Act
  Annex III; Malaysia Personal Data Protection Act 2010.